# Bank Branch Cash-Out Forecasting

This notebook contains the end-to-end workflow used in an Independent Study project
to forecast next-day total cash-out amounts for bank branches.

> **Confidentiality:** The original banking dataset is not included in this repository.
> Notebook outputs were intentionally cleared because they may contain real transaction values.

### Main workflow
1. Data validation and cleaning
2. Missing branch-date audit
3. Daily branch-date grid construction
4. Next-day target creation
5. Calendar, lag, and rolling features
6. Chronological Subtrain / Validation / Final Test split
7. Baseline and machine-learning model comparison
8. Residual modeling with HistGradientBoosting
9. Final-test evaluation
10. Safety-buffer simulation


In [ ]:
# ติดตั้ง Library ที่จำเป็น
!pip install -q holidays xgboost

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

# ตั้งค่าการแสดงตัวเลข
pd.options.display.float_format = "{:,.2f}".format

# ใช้ค่าเดียวกันทุกครั้ง เพื่อให้ผลทำซ้ำได้
RANDOM_STATE = 42

# False = รันผล Final จริง
# อย่าเปลี่ยนเป็น True ในรอบเก็บผลสอบ
FAST_MODE = False

# โฟลเดอร์สำหรับเก็บผลลัพธ์
OUTPUT_DIR = "strict_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("ติดตั้งและ Import Library สำเร็จ")
print("FAST_MODE =", FAST_MODE)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
from google.colab import files
import io

# อัปโหลดไฟล์ข้อมูลดิบ
uploaded = files.upload()

# ต้องอัปโหลดครั้งละ 1 ไฟล์
if len(uploaded) != 1:
    raise ValueError("กรุณาอัปโหลดข้อมูลครั้งละ 1 ไฟล์")

file_name = next(iter(uploaded))
file_bytes = uploaded[file_name]

# อ่านไฟล์ตามนามสกุล
if file_name.lower().endswith(".csv"):
    try:
        df_raw = pd.read_csv(io.BytesIO(file_bytes))
    except UnicodeDecodeError:
        df_raw = pd.read_csv(
            io.BytesIO(file_bytes),
            encoding="tis-620"
        )

elif file_name.lower().endswith((".xlsx", ".xls")):
    df_raw = pd.read_excel(io.BytesIO(file_bytes))

else:
    raise ValueError("รองรับเฉพาะไฟล์ CSV หรือ Excel")

# ลบช่องว่างที่อาจติดมากับชื่อคอลัมน์
df_raw.columns = df_raw.columns.astype(str).str.strip()

print("=" * 70)
print("ชื่อไฟล์:", file_name)
print("จำนวนแถว:", f"{df_raw.shape[0]:,}")
print("จำนวนคอลัมน์:", df_raw.shape[1])
print("=" * 70)

print("\nรายชื่อคอลัมน์:")
for number, column in enumerate(df_raw.columns, start=1):
    print(f"{number:>2}. {column}")

print("\nตัวอย่างข้อมูล 5 แถวแรก:")
display(df_raw.head())

In [ ]:
# ============================================================
# STEP 3: Clean raw data and create aggregate cash variables
# ============================================================

df = df_raw.copy()

# 1) ตรวจสอบคอลัมน์ที่จำเป็น
required_cols = [
    "EFFECTIVE_DATE",
    "BRANCH_CODE",
    "CASH_IN_AMT",
    "CASH_OUT_AMT",
    "CASH_IN_AMT_GT1M",
    "CASH_OUT_AMT_GT1M",
    "SHIPIN1K",
    "SHIPIN500",
    "SHIPIN100",
    "SHIPIN_OTHER",
    "SHIPOUT1K",
    "SHIPOUT500",
    "SHIPOUT100",
    "SHIPOUT_OTHER",
    "CASHSTOCK1K",
    "CASHSTOCK500",
    "CASHSTOCK100",
    "CASHSTOCK_OTHER"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"ไม่พบคอลัมน์ที่จำเป็น: {missing_cols}"
    )

print("ตรวจสอบคอลัมน์ที่จำเป็น: ผ่าน")


# 2) แปลงวันที่และรหัสสาขา
df["EFFECTIVE_DATE"] = pd.to_datetime(
    df["EFFECTIVE_DATE"],
    errors="coerce"
)

df["BRANCH_CODE"] = (
    df["BRANCH_CODE"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# 3) แปลงคอลัมน์ตัวเลข
numeric_cols = [
    "CASH_IN_AMT",
    "CASH_OUT_AMT",
    "CASH_IN_AMT_GT1M",
    "CASH_OUT_AMT_GT1M",
    "SHIPIN1K",
    "SHIPIN500",
    "SHIPIN100",
    "SHIPIN_OTHER",
    "SHIPOUT1K",
    "SHIPOUT500",
    "SHIPOUT100",
    "SHIPOUT_OTHER",
    "CASHSTOCK1K",
    "CASHSTOCK500",
    "CASHSTOCK100",
    "CASHSTOCK_OTHER"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# 4) ตรวจสอบข้อมูลผิดปกติก่อนสร้างตัวแปร
invalid_date_count = (
    df["EFFECTIVE_DATE"]
    .isna()
    .sum()
)

missing_branch_count = (
    df["BRANCH_CODE"]
    .isin(["", "NAN", "NONE"])
    .sum()
)

missing_numeric_count = (
    df[numeric_cols]
    .isna()
    .sum()
    .sum()
)

duplicate_count = (
    df.duplicated(
        subset=[
            "EFFECTIVE_DATE",
            "BRANCH_CODE"
        ]
    )
    .sum()
)

# ตรวจค่าติดลบในคอลัมน์จำนวนเงินต้นฉบับ
negative_mask = (
    df[numeric_cols] < 0
)

negative_count_by_column = (
    negative_mask
    .sum()
)

negative_columns = (
    negative_count_by_column[
        negative_count_by_column > 0
    ]
)

negative_row_mask = (
    negative_mask
    .any(axis=1)
)

negative_row_count = (
    negative_row_mask
    .sum()
)

print("\nผลการตรวจสอบข้อมูลดิบ")
print("-" * 55)
print(
    "วันที่อ่านไม่ได้:",
    f"{invalid_date_count:,}"
)
print(
    "รหัสสาขาหาย:",
    f"{missing_branch_count:,}"
)
print(
    "ค่าตัวเลขที่หายทั้งหมด:",
    f"{missing_numeric_count:,}"
)
print(
    "ข้อมูลสาขา-วันที่ซ้ำ:",
    f"{duplicate_count:,}"
)
print(
    "แถวที่มีค่าติดลบ:",
    f"{negative_row_count:,}"
)


# แสดงรายละเอียดเมื่อตรวจพบค่าติดลบ
if negative_row_count > 0:

    negative_summary = pd.DataFrame({
        "จำนวนค่าติดลบ":
            negative_columns,
        "ค่าต่ำสุด":
            df[
                negative_columns.index
            ].min()
    })

    print("\nสรุปคอลัมน์ที่มีค่าติดลบ")
    display(negative_summary)

    print("\nตัวอย่างแถวที่มีค่าติดลบ")

    display(
        df.loc[
            negative_row_mask,
            [
                "EFFECTIVE_DATE",
                "BRANCH_CODE"
            ] + numeric_cols
        ]
        .sort_values(
            [
                "EFFECTIVE_DATE",
                "BRANCH_CODE"
            ]
        )
        .head(50)
    )


# หยุดทันทีถ้ามีปัญหาที่อาจกระทบการคำนวณ
if invalid_date_count > 0:
    raise ValueError(
        "พบวันที่ที่แปลงไม่ได้ "
        "ต้องตรวจสอบก่อนดำเนินการ"
    )

if missing_branch_count > 0:
    raise ValueError(
        "พบข้อมูลที่ไม่มีรหัสสาขา "
        "ต้องตรวจสอบก่อนดำเนินการ"
    )

if missing_numeric_count > 0:
    raise ValueError(
        "พบค่าตัวเลขที่หาย "
        "ยังไม่ควรแทนด้วย 0 โดยอัตโนมัติ"
    )

if duplicate_count > 0:
    raise ValueError(
        "พบข้อมูลสาขา-วันที่ซ้ำ "
        "ต้องตรวจสอบก่อนสร้าง Time Series"
    )

if negative_row_count > 0:
    raise ValueError(
        "พบค่าติดลบในข้อมูลต้นฉบับ "
        "ต้องตรวจสอบว่าเป็น reversal "
        "หรือความผิดพลาดก่อนดำเนินการ"
    )


# 5) สร้างยอดรวมหลัก
df["TOTAL_CASH_IN"] = (
    df["CASH_IN_AMT"]
    + df["CASH_IN_AMT_GT1M"]
)

df["TOTAL_CASH_OUT"] = (
    df["CASH_OUT_AMT"]
    + df["CASH_OUT_AMT_GT1M"]
)

df["TOTAL_SHIPIN"] = df[
    [
        "SHIPIN1K",
        "SHIPIN500",
        "SHIPIN100",
        "SHIPIN_OTHER"
    ]
].sum(axis=1)

df["TOTAL_SHIPOUT"] = df[
    [
        "SHIPOUT1K",
        "SHIPOUT500",
        "SHIPOUT100",
        "SHIPOUT_OTHER"
    ]
].sum(axis=1)

df["TOTAL_CASH_STOCK"] = df[
    [
        "CASHSTOCK1K",
        "CASHSTOCK500",
        "CASHSTOCK100",
        "CASHSTOCK_OTHER"
    ]
].sum(axis=1)


# 6) เรียงข้อมูลตามสาขาและวันที่
df = (
    df.sort_values(
        [
            "BRANCH_CODE",
            "EFFECTIVE_DATE"
        ]
    )
    .reset_index(drop=True)
)


# 7) สรุปข้อมูลหลังทำความสะอาด
print("\n" + "=" * 60)
print("ทำความสะอาดและสร้างยอดรวมสำเร็จ")
print("=" * 60)

print(
    "จำนวนแถว:",
    f"{len(df):,}"
)

print(
    "จำนวนสาขา:",
    df["BRANCH_CODE"].nunique()
)

print(
    "จำนวนวันที่ไม่ซ้ำ:",
    df["EFFECTIVE_DATE"].nunique()
)

print(
    "ช่วงวันที่:",
    df["EFFECTIVE_DATE"].min().date(),
    "ถึง",
    df["EFFECTIVE_DATE"].max().date()
)

print("\nรายชื่อสาขา:")
print(
    sorted(
        df["BRANCH_CODE"].unique()
    )
)

display(
    df[
        [
            "EFFECTIVE_DATE",
            "BRANCH_CODE",
            "TOTAL_CASH_IN",
            "TOTAL_CASH_OUT",
            "TOTAL_SHIPIN",
            "TOTAL_SHIPOUT",
            "TOTAL_CASH_STOCK"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# STEP 4: Audit missing branch-date combinations
# ============================================================

# ช่วงวันทั้งหมดในข้อมูล
all_dates = pd.date_range(
    start=df["EFFECTIVE_DATE"].min(),
    end=df["EFFECTIVE_DATE"].max(),
    freq="D"
)

all_branches = sorted(df["BRANCH_CODE"].unique())

# สร้างตารางสาขา × วันที่ที่ควรมีครบ
expected_index = pd.MultiIndex.from_product(
    [all_branches, all_dates],
    names=["BRANCH_CODE", "EFFECTIVE_DATE"]
)

# ตารางสาขา × วันที่ที่มีอยู่จริง
actual_index = pd.MultiIndex.from_frame(
    df[["BRANCH_CODE", "EFFECTIVE_DATE"]]
)

# หาคู่สาขา-วันที่ที่หายไป
missing_index = expected_index.difference(actual_index)

missing_branch_dates = (
    missing_index
    .to_frame(index=False)
    .sort_values(["BRANCH_CODE", "EFFECTIVE_DATE"])
    .reset_index(drop=True)
)

# สรุปจำนวนแถวของแต่ละสาขา
branch_date_summary = (
    df.groupby("BRANCH_CODE")
      .agg(
          ACTUAL_ROWS=("EFFECTIVE_DATE", "size"),
          FIRST_DATE=("EFFECTIVE_DATE", "min"),
          LAST_DATE=("EFFECTIVE_DATE", "max")
      )
      .reset_index()
)

branch_date_summary["EXPECTED_ROWS"] = len(all_dates)
branch_date_summary["MISSING_ROWS"] = (
    branch_date_summary["EXPECTED_ROWS"]
    - branch_date_summary["ACTUAL_ROWS"]
)

print("=" * 65)
print("ตรวจสอบความต่อเนื่องของข้อมูลรายวัน")
print("=" * 65)
print("จำนวนวันที่ที่ควรมีต่อสาขา:", f"{len(all_dates):,}")
print(
    "จำนวนแถวที่ควรมีทั้งหมด:",
    f"{len(expected_index):,}"
)
print("จำนวนแถวจริง:", f"{len(df):,}")
print(
    "จำนวนสาขา-วันที่ที่ขาด:",
    f"{len(missing_branch_dates):,}"
)

print("\nสรุปรายสาขา:")
display(branch_date_summary)

print("\nรายการสาขา-วันที่ที่ขาด:")
display(missing_branch_dates)

In [ ]:
# ============================================================
# STEP 5: Complete daily branch-date grid without inventing values
# ============================================================

# เก็บเครื่องหมายว่าแถวเดิมมีข้อมูลจริง
df["IS_OBSERVED"] = 1

# สร้างตารางครบทุกสาขา × ทุกวัน
full_index = pd.MultiIndex.from_product(
    [all_branches, all_dates],
    names=["BRANCH_CODE", "EFFECTIVE_DATE"]
)

df_daily = (
    df.set_index(["BRANCH_CODE", "EFFECTIVE_DATE"])
      .reindex(full_index)
      .reset_index()
      .sort_values(["BRANCH_CODE", "EFFECTIVE_DATE"])
      .reset_index(drop=True)
)

# แถวที่เพิ่มเข้ามาใหม่ถือว่าไม่มีข้อมูลจริง
df_daily["IS_OBSERVED"] = (
    df_daily["IS_OBSERVED"]
    .fillna(0)
    .astype(int)
)

df_daily["IS_MISSING_RECORD"] = (
    1 - df_daily["IS_OBSERVED"]
)

# BRANCH_NAME เป็นข้อมูลประจำสาขา จึงเติมจากแถวอื่นของสาขาเดียวกันได้
df_daily["BRANCH_NAME"] = (
    df_daily.groupby("BRANCH_CODE")["BRANCH_NAME"]
            .transform(lambda s: s.ffill().bfill())
)

# สำคัญ: ไม่เติมค่าธุรกรรมที่หายด้วย 0
# ค่าของ 8 แถวที่ขาดจะยังเป็น NaN

# ตรวจสอบความต่อเนื่องรายวัน
date_gap_check = (
    df_daily.groupby("BRANCH_CODE")["EFFECTIVE_DATE"]
            .diff()
            .dropna()
            .ne(pd.Timedelta(days=1))
            .sum()
)

inserted_rows = df_daily[
    df_daily["IS_MISSING_RECORD"] == 1
].copy()

print("=" * 70)
print("สร้างโครงข้อมูลรายวันครบทุกสาขาสำเร็จ")
print("=" * 70)

print("จำนวนแถวหลังจัดปฏิทิน:", f"{len(df_daily):,}")
print(
    "จำนวนแถวที่มีข้อมูลจริง:",
    f"{df_daily['IS_OBSERVED'].sum():,}"
)
print(
    "จำนวนแถวที่เพิ่มเพื่อให้วันที่ต่อเนื่อง:",
    f"{df_daily['IS_MISSING_RECORD'].sum():,}"
)
print("จำนวนช่องว่างของวันที่หลังจัดใหม่:", date_gap_check)

print("\nแถวที่เพิ่มเข้ามา:")
display(
    inserted_rows[
        [
            "EFFECTIVE_DATE",
            "BRANCH_CODE",
            "IS_OBSERVED",
            "IS_MISSING_RECORD",
            "TOTAL_CASH_IN",
            "TOTAL_CASH_OUT",
            "TOTAL_CASH_STOCK"
        ]
    ]
)

print("\nตัวอย่างข้อมูลรอบช่วงวันที่ขาด:")
display(
    df_daily[
        (df_daily["BRANCH_CODE"].isin(["A1", "A2", "A3", "A4"]))
        & (df_daily["EFFECTIVE_DATE"].between(
            "2018-07-12",
            "2018-07-17"
        ))
    ][
        [
            "EFFECTIVE_DATE",
            "BRANCH_CODE",
            "IS_OBSERVED",
            "TOTAL_CASH_OUT",
            "TOTAL_CASH_STOCK"
        ]
    ]
)

In [ ]:
# ============================================================
# STEP 6: Create next-calendar-day target within each branch
# ============================================================

df_target = df_daily.copy()

# ต้องเรียงตามสาขาและวันก่อนใช้ shift
df_target = (
    df_target
    .sort_values(["BRANCH_CODE", "EFFECTIVE_DATE"])
    .reset_index(drop=True)
)

# วันที่ที่ต้องการพยากรณ์
df_target["FORECAST_DATE"] = (
    df_target["EFFECTIVE_DATE"] + pd.Timedelta(days=1)
)

# ยอดเงินสดออกของวันถัดไปในสาขาเดียวกัน
df_target["TARGET_T_PLUS_1"] = (
    df_target
    .groupby("BRANCH_CODE")["TOTAL_CASH_OUT"]
    .shift(-1)
)

# ตรวจว่าวันถัดไปเป็นข้อมูลจริงหรือเป็นแถวที่เติมโครงปฏิทิน
df_target["TARGET_IS_OBSERVED"] = (
    df_target
    .groupby("BRANCH_CODE")["IS_OBSERVED"]
    .shift(-1)
)

# วันที่ของ record ถัดไป ใช้ตรวจว่าเป็นวันตามปฏิทินจริง
df_target["ACTUAL_NEXT_DATE"] = (
    df_target
    .groupby("BRANCH_CODE")["EFFECTIVE_DATE"]
    .shift(-1)
)

df_target["IS_NEXT_CALENDAR_DAY"] = (
    df_target["ACTUAL_NEXT_DATE"]
    == df_target["FORECAST_DATE"]
).astype(int)

# แถวที่ใช้เป็นฐานสำหรับสร้างโมเดลได้
# 1) วันปัจจุบันต้องมีข้อมูลจริง
# 2) วันถัดไปต้องมีข้อมูลจริง
# 3) ต้องมี Target
# 4) record ถัดไปต้องเป็นวันตามปฏิทินจริง
df_target["BASE_ROW_ELIGIBLE"] = (
    (df_target["IS_OBSERVED"] == 1)
    & (df_target["TARGET_IS_OBSERVED"] == 1)
    & (df_target["TARGET_T_PLUS_1"].notna())
    & (df_target["IS_NEXT_CALENDAR_DAY"] == 1)
).astype(int)

# ตรวจสอบ Target ด้วยการ join ตาม FORECAST_DATE โดยตรง
target_direct_check = (
    df_target[
        ["BRANCH_CODE", "EFFECTIVE_DATE", "TOTAL_CASH_OUT"]
    ]
    .rename(columns={
        "EFFECTIVE_DATE": "FORECAST_DATE",
        "TOTAL_CASH_OUT": "TARGET_DIRECT_CHECK"
    })
)

df_target = df_target.merge(
    target_direct_check,
    on=["BRANCH_CODE", "FORECAST_DATE"],
    how="left",
    validate="many_to_one"
)

# เปรียบเทียบ Target ที่สร้างด้วย shift กับการจับคู่วันที่โดยตรง
target_match_mask = (
    df_target["TARGET_T_PLUS_1"].notna()
    & df_target["TARGET_DIRECT_CHECK"].notna()
)

target_mismatch_count = (
    ~np.isclose(
        df_target.loc[target_match_mask, "TARGET_T_PLUS_1"],
        df_target.loc[target_match_mask, "TARGET_DIRECT_CHECK"],
        rtol=1e-10,
        atol=1e-8
    )
).sum()

# สรุปผล
print("=" * 72)
print("สร้าง Target สำหรับพยากรณ์วันถัดไปสำเร็จ")
print("=" * 72)

print("จำนวนแถวทั้งหมด:", f"{len(df_target):,}")
print(
    "จำนวนแถวที่ Target มีข้อมูล:",
    f"{df_target['TARGET_T_PLUS_1'].notna().sum():,}"
)
print(
    "จำนวนแถวฐานที่ใช้สร้างโมเดลได้:",
    f"{df_target['BASE_ROW_ELIGIBLE'].sum():,}"
)
print(
    "จำนวน Target ที่ไม่ตรงกับการจับคู่วันที่โดยตรง:",
    f"{target_mismatch_count:,}"
)

if target_mismatch_count > 0:
    raise ValueError(
        "พบ Target ที่ไม่ตรงกับยอดเงินสดออกของวันถัดไป"
    )

print("\nเหตุผลที่แถวไม่สามารถใช้เป็นฐานโมเดล:")
print(
    "วันปัจจุบันไม่มีข้อมูลจริง:",
    f"{(df_target['IS_OBSERVED'] == 0).sum():,}"
)
print(
    "วันถัดไปไม่มีข้อมูลจริง:",
    f"{(df_target['TARGET_IS_OBSERVED'] == 0).sum():,}"
)
print(
    "ไม่มี Target เช่น วันสุดท้ายของแต่ละสาขา:",
    f"{df_target['TARGET_T_PLUS_1'].isna().sum():,}"
)

print("\nตัวอย่างการจับคู่วัน t กับวัน t+1:")
display(
    df_target.loc[
        df_target["BASE_ROW_ELIGIBLE"] == 1,
        [
            "BRANCH_CODE",
            "EFFECTIVE_DATE",
            "TOTAL_CASH_OUT",
            "FORECAST_DATE",
            "TARGET_T_PLUS_1",
            "BASE_ROW_ELIGIBLE"
        ]
    ].head(15)
)
print("\nตรวจช่วงวันที่ข้อมูลขาด:")

display(
    df_target[
        (df_target["BRANCH_CODE"].isin(["A1", "A2", "A3", "A4"]))
        & (
            df_target["EFFECTIVE_DATE"].between(
                "2018-07-12",
                "2018-07-16"
            )
        )
    ][
        [
            "BRANCH_CODE",
            "EFFECTIVE_DATE",
            "IS_OBSERVED",
            "TOTAL_CASH_OUT",
            "FORECAST_DATE",
            "TARGET_IS_OBSERVED",
            "TARGET_T_PLUS_1",
            "BASE_ROW_ELIGIBLE"
        ]
    ]
)

In [ ]:
# ============================================================
# STEP 7: Create forecast-date calendar features
# ============================================================

df_features = df_target.copy()

# วันที่ที่เราต้องการพยากรณ์จริง
forecast_date = df_features["FORECAST_DATE"]

# ------------------------------------------------------------
# 1) ตัวแปรพื้นฐานจากวันที่พยากรณ์
# ------------------------------------------------------------
df_features["FORECAST_YEAR"] = forecast_date.dt.year
df_features["FORECAST_MONTH"] = forecast_date.dt.month
df_features["FORECAST_DAY"] = forecast_date.dt.day

# Monday = 0, Tuesday = 1, ..., Sunday = 6
df_features["FORECAST_DAY_OF_WEEK"] = forecast_date.dt.dayofweek

df_features["FORECAST_IS_WEEKEND"] = (
    df_features["FORECAST_DAY_OF_WEEK"].isin([5, 6])
).astype(int)

df_features["FORECAST_IS_MONTH_START"] = (
    forecast_date.dt.is_month_start
).astype(int)

df_features["FORECAST_IS_MONTH_END"] = (
    forecast_date.dt.is_month_end
).astype(int)

df_features["FORECAST_QUARTER"] = forecast_date.dt.quarter


# ------------------------------------------------------------
# 2) วันหยุดประเทศไทย
# ------------------------------------------------------------
holiday_years = sorted(
    df_features["FORECAST_DATE"]
    .dropna()
    .dt.year
    .unique()
    .astype(int)
)

thai_holidays = holidays.country_holidays(
    "TH",
    years=holiday_years
)

# ใช้ date object เพื่อเปรียบเทียบอย่างชัดเจน
thai_holiday_dates = set(thai_holidays.keys())

forecast_date_only = forecast_date.dt.date
next_forecast_date_only = (
    forecast_date + pd.Timedelta(days=1)
).dt.date

previous_forecast_date_only = (
    forecast_date - pd.Timedelta(days=1)
).dt.date

# วันที่พยากรณ์เป็นวันหยุด
df_features["FORECAST_IS_HOLIDAY"] = (
    forecast_date_only.isin(thai_holiday_dates)
).astype(int)

# วันที่พยากรณ์เป็นวันก่อนวันหยุด
df_features["FORECAST_IS_PRE_HOLIDAY"] = (
    next_forecast_date_only.isin(thai_holiday_dates)
).astype(int)

# วันที่พยากรณ์เป็นวันหลังวันหยุด
df_features["FORECAST_IS_POST_HOLIDAY"] = (
    previous_forecast_date_only.isin(thai_holiday_dates)
).astype(int)


# ------------------------------------------------------------
# 3) ตัวแปรช่วงเงินเดือน
# ------------------------------------------------------------
# สมมติฐานของงาน:
# ช่วงปลายเดือนวันที่ 25-31 และต้นเดือนวันที่ 1-2
PAYDAY_DAYS = [
    25, 26, 27, 28, 29, 30, 31,
    1, 2
]

df_features["FORECAST_IS_PAYDAY_PERIOD"] = (
    df_features["FORECAST_DAY"].isin(PAYDAY_DAYS)
).astype(int)

df_features["FORECAST_IS_NEAR_MONTH_END"] = (
    df_features["FORECAST_DAY"].isin([28, 29, 30, 31])
).astype(int)

df_features["FORECAST_IS_NEAR_MONTH_START"] = (
    df_features["FORECAST_DAY"].isin([1, 2, 3])
).astype(int)


# ------------------------------------------------------------
# 4) ตรวจสอบความถูกต้อง
# ------------------------------------------------------------
calendar_feature_cols = [
    "FORECAST_YEAR",
    "FORECAST_MONTH",
    "FORECAST_DAY",
    "FORECAST_DAY_OF_WEEK",
    "FORECAST_QUARTER",
    "FORECAST_IS_WEEKEND",
    "FORECAST_IS_MONTH_START",
    "FORECAST_IS_MONTH_END",
    "FORECAST_IS_HOLIDAY",
    "FORECAST_IS_PRE_HOLIDAY",
    "FORECAST_IS_POST_HOLIDAY",
    "FORECAST_IS_PAYDAY_PERIOD",
    "FORECAST_IS_NEAR_MONTH_END",
    "FORECAST_IS_NEAR_MONTH_START"
]

calendar_missing_count = (
    df_features[calendar_feature_cols]
    .isna()
    .sum()
    .sum()
)

binary_calendar_cols = [
    "FORECAST_IS_WEEKEND",
    "FORECAST_IS_MONTH_START",
    "FORECAST_IS_MONTH_END",
    "FORECAST_IS_HOLIDAY",
    "FORECAST_IS_PRE_HOLIDAY",
    "FORECAST_IS_POST_HOLIDAY",
    "FORECAST_IS_PAYDAY_PERIOD",
    "FORECAST_IS_NEAR_MONTH_END",
    "FORECAST_IS_NEAR_MONTH_START"
]

invalid_binary_count = 0

for col in binary_calendar_cols:
    invalid_binary_count += (
        ~df_features[col].isin([0, 1])
    ).sum()


# FORECAST_DATE ต้องเท่ากับ EFFECTIVE_DATE + 1 วันเสมอ
forecast_date_mismatch = (
    df_features["FORECAST_DATE"]
    != df_features["EFFECTIVE_DATE"] + pd.Timedelta(days=1)
).sum()


print("=" * 72)
print("สร้างตัวแปรปฏิทินจากวันที่พยากรณ์สำเร็จ")
print("=" * 72)

print("ปีวันหยุดที่โหลด:", holiday_years)
print("จำนวนวันหยุดในปฏิทิน:", len(thai_holiday_dates))
print("ค่าที่หายใน Calendar Features:", calendar_missing_count)
print("ค่า Binary ที่ไม่ใช่ 0 หรือ 1:", invalid_binary_count)
print(
    "FORECAST_DATE ไม่เท่ากับ EFFECTIVE_DATE + 1 วัน:",
    forecast_date_mismatch
)

print("\nจำนวน Branch-day records ตามประเภทวัน:")
print(
    df_features[
        [
            "FORECAST_IS_WEEKEND",
            "FORECAST_IS_HOLIDAY",
            "FORECAST_IS_PRE_HOLIDAY",
            "FORECAST_IS_POST_HOLIDAY",
            "FORECAST_IS_PAYDAY_PERIOD"
        ]
    ].sum()
)

print("\nตัวอย่าง Calendar Features:")
display(
    df_features[
        [
            "EFFECTIVE_DATE",
            "FORECAST_DATE",
            "BRANCH_CODE",
            "FORECAST_DAY_OF_WEEK",
            "FORECAST_MONTH",
            "FORECAST_DAY",
            "FORECAST_IS_WEEKEND",
            "FORECAST_IS_HOLIDAY",
            "FORECAST_IS_PRE_HOLIDAY",
            "FORECAST_IS_POST_HOLIDAY",
            "FORECAST_IS_PAYDAY_PERIOD"
        ]
    ].head(15)
)

print("\nตัวอย่างวันที่พยากรณ์ที่เป็นวันหยุด:")
display(
    df_features.loc[
        df_features["FORECAST_IS_HOLIDAY"] == 1,
        [
            "EFFECTIVE_DATE",
            "FORECAST_DATE",
            "BRANCH_CODE",
            "FORECAST_DAY_OF_WEEK",
            "FORECAST_IS_HOLIDAY"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# STEP 8: Create leakage-safe lag and rolling features
# ============================================================

df_features = (
    df_features
    .sort_values(["BRANCH_CODE", "EFFECTIVE_DATE"])
    .reset_index(drop=True)
)

# ตัวแปรธุรกรรมที่ทราบแล้ว ณ สิ้นวัน t
base_numeric_features = [
    "TOTAL_CASH_IN",
    "TOTAL_CASH_OUT",
    "TOTAL_SHIPIN",
    "TOTAL_SHIPOUT",
    "TOTAL_CASH_STOCK"
]

# Lag หมายถึงย้อนหลังจากวัน t
lag_periods = [1, 2, 3, 7, 14, 28]

# Rolling รวมวันปัจจุบัน t
# ใช้เป็นข้อมูลสำหรับพยากรณ์วัน t+1
rolling_windows = [3, 7, 14, 30]

lag_feature_cols = []
rolling_feature_cols = []

# ------------------------------------------------------------
# 1) สร้าง Lag แยกตามสาขา
# ------------------------------------------------------------
for col in base_numeric_features:
    grouped_series = df_features.groupby("BRANCH_CODE")[col]

    for lag in lag_periods:
        feature_name = f"{col}_LAG{lag}"

        df_features[feature_name] = grouped_series.shift(lag)

        lag_feature_cols.append(feature_name)


# ------------------------------------------------------------
# 2) สร้าง Rolling Mean แยกตามสาขา
# ------------------------------------------------------------
# ไม่ใช้ shift เพราะค่าของวัน t ทราบแล้ว
# และเรากำลังพยากรณ์วัน t+1
#
# min_periods=window หมายถึงต้องมีข้อมูลจริงครบทั้งช่วง
# หากมีวันที่ขาด จะไม่คำนวณค่าเฉลี่ยโดยข้ามวันนั้น
# ------------------------------------------------------------
for col in base_numeric_features:

    for window in rolling_windows:
        feature_name = f"{col}_ROLLING{window}"

        df_features[feature_name] = (
            df_features
            .groupby("BRANCH_CODE")[col]
            .transform(
                lambda series: series.rolling(
                    window=window,
                    min_periods=window
                ).mean()
            )
        )

        rolling_feature_cols.append(feature_name)


# ------------------------------------------------------------
# 3) สร้าง Rolling Standard Deviation สำหรับยอด Cash Out
# ------------------------------------------------------------
# ใช้ช่วยสะท้อนความผันผวนของเงินสดออกในอดีต
# ddof=0 เพื่อให้คำนวณแบบ population standard deviation
# ------------------------------------------------------------
for window in [7, 14, 30]:
    feature_name = f"TOTAL_CASH_OUT_ROLLING_STD{window}"

    df_features[feature_name] = (
        df_features
        .groupby("BRANCH_CODE")["TOTAL_CASH_OUT"]
        .transform(
            lambda series: series.rolling(
                window=window,
                min_periods=window
            ).std(ddof=0)
        )
    )

    rolling_feature_cols.append(feature_name)


# ------------------------------------------------------------
# 4) เลือกเฉพาะแถวที่ Target และข้อมูลวันปัจจุบันใช้ได้
# ------------------------------------------------------------
df_model = df_features.loc[
    df_features["BASE_ROW_ELIGIBLE"] == 1
].copy()

history_feature_cols = (
    lag_feature_cols
    + rolling_feature_cols
)

# ตรวจว่า Lag/Rolling พร้อมครบทุกตัว
df_model["HAS_COMPLETE_HISTORY"] = (
    df_model[history_feature_cols]
    .notna()
    .all(axis=1)
    .astype(int)
)

rows_before_history_filter = len(df_model)

df_model = (
    df_model.loc[
        df_model["HAS_COMPLETE_HISTORY"] == 1
    ]
    .sort_values(["FORECAST_DATE", "BRANCH_CODE"])
    .reset_index(drop=True)
)

rows_after_history_filter = len(df_model)


# ------------------------------------------------------------
# 5) ตรวจสอบด้วยการคำนวณ Rolling30 ซ้ำด้วยมือ
# ------------------------------------------------------------
audit_sample = df_model.sample(
    n=min(20, len(df_model)),
    random_state=RANDOM_STATE
)

rolling30_mismatch = 0
lag1_mismatch = 0
future_data_violation = 0

for _, row in audit_sample.iterrows():

    branch = row["BRANCH_CODE"]
    current_date = row["EFFECTIVE_DATE"]

    branch_history = df_features.loc[
        (df_features["BRANCH_CODE"] == branch)
        & (
            df_features["EFFECTIVE_DATE"].between(
                current_date - pd.Timedelta(days=29),
                current_date
            )
        ),
        ["EFFECTIVE_DATE", "TOTAL_CASH_OUT"]
    ].sort_values("EFFECTIVE_DATE")

    # Rolling30 ต้องใช้วัน t-29 ถึงวัน t รวม 30 วัน
    manual_rolling30 = branch_history["TOTAL_CASH_OUT"].mean()

    stored_rolling30 = row["TOTAL_CASH_OUT_ROLLING30"]

    if not np.isclose(
        manual_rolling30,
        stored_rolling30,
        rtol=1e-10,
        atol=1e-6
    ):
        rolling30_mismatch += 1

    # ตรวจ LAG1 ต้องตรงกับยอดวันที่ t-1
    previous_day_value = df_features.loc[
        (df_features["BRANCH_CODE"] == branch)
        & (
            df_features["EFFECTIVE_DATE"]
            == current_date - pd.Timedelta(days=1)
        ),
        "TOTAL_CASH_OUT"
    ]

    if len(previous_day_value) != 1:
        lag1_mismatch += 1

    elif not np.isclose(
        previous_day_value.iloc[0],
        row["TOTAL_CASH_OUT_LAG1"],
        rtol=1e-10,
        atol=1e-6
    ):
        lag1_mismatch += 1

    # วันที่สูงสุดที่ใช้สร้าง Rolling ต้องไม่เกินวัน t
    if branch_history["EFFECTIVE_DATE"].max() > current_date:
        future_data_violation += 1


# ------------------------------------------------------------
# 6) สรุปผล
# ------------------------------------------------------------
print("=" * 75)
print("สร้าง Lag และ Rolling Features สำเร็จ")
print("=" * 75)

print("จำนวน Lag Features:", len(lag_feature_cols))
print("จำนวน Rolling Features:", len(rolling_feature_cols))

print(
    "แถวก่อนตรวจประวัติย้อนหลัง:",
    f"{rows_before_history_filter:,}"
)

print(
    "แถวที่มีประวัติย้อนหลังครบ:",
    f"{rows_after_history_filter:,}"
)

print(
    "แถวที่ถูกตัดเพราะประวัติไม่ครบ:",
    f"{rows_before_history_filter - rows_after_history_filter:,}"
)

print("\nผลตรวจ Data Leakage จากตัวอย่าง:")
print("Rolling30 คำนวณไม่ตรง:", rolling30_mismatch)
print("LAG1 คำนวณไม่ตรง:", lag1_mismatch)
print("พบการใช้วันที่อนาคต:", future_data_violation)

print("\nช่วงวันที่ของข้อมูลพร้อมสร้างโมเดล:")
print(
    df_model["EFFECTIVE_DATE"].min().date(),
    "ถึง",
    df_model["EFFECTIVE_DATE"].max().date()
)

print("\nช่วงวันที่ที่ต้องการพยากรณ์:")
print(
    df_model["FORECAST_DATE"].min().date(),
    "ถึง",
    df_model["FORECAST_DATE"].max().date()
)

print("\nตัวอย่าง Lag และ Rolling:")
display(
    df_model[
        [
            "BRANCH_CODE",
            "EFFECTIVE_DATE",
            "FORECAST_DATE",
            "TOTAL_CASH_OUT",
            "TOTAL_CASH_OUT_LAG1",
            "TOTAL_CASH_OUT_LAG7",
            "TOTAL_CASH_OUT_ROLLING7",
            "TOTAL_CASH_OUT_ROLLING30",
            "TARGET_T_PLUS_1"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# STEP 9: Build model-ready feature matrix
# ============================================================

# ------------------------------------------------------------
# 1) ตัวแปรข้อมูล ณ สิ้นวัน t
# ใช้พยากรณ์ยอดเงินสดออกของวัน t+1
# ------------------------------------------------------------
current_day_feature_cols = [
    "TOTAL_CASH_IN",
    "TOTAL_CASH_OUT",
    "TOTAL_SHIPIN",
    "TOTAL_SHIPOUT",
    "TOTAL_CASH_STOCK"
]


# ------------------------------------------------------------
# 2) ตัวแปรปฏิทินของวันที่ต้องการพยากรณ์ t+1
# ------------------------------------------------------------
calendar_numeric_cols = [
    "FORECAST_YEAR",
    "FORECAST_DAY"
]

calendar_categorical_cols = [
    "FORECAST_DAY_OF_WEEK",
    "FORECAST_MONTH"
]

calendar_binary_cols = [
    "FORECAST_IS_WEEKEND",
    "FORECAST_IS_MONTH_START",
    "FORECAST_IS_MONTH_END",
    "FORECAST_IS_HOLIDAY",
    "FORECAST_IS_PRE_HOLIDAY",
    "FORECAST_IS_POST_HOLIDAY",
    "FORECAST_IS_PAYDAY_PERIOD",
    "FORECAST_IS_NEAR_MONTH_END",
    "FORECAST_IS_NEAR_MONTH_START"
]


# ------------------------------------------------------------
# 3) รวมตัวแปรทั้งหมดก่อน One-hot Encoding
# ------------------------------------------------------------
numeric_feature_cols = (
    current_day_feature_cols
    + calendar_numeric_cols
    + calendar_binary_cols
    + lag_feature_cols
    + rolling_feature_cols
)

categorical_feature_cols = [
    "BRANCH_CODE",
    "FORECAST_DAY_OF_WEEK",
    "FORECAST_MONTH"
]

all_raw_feature_cols = (
    numeric_feature_cols
    + categorical_feature_cols
)


# ------------------------------------------------------------
# 4) เลือกเฉพาะคอลัมน์ที่จำเป็น
# ------------------------------------------------------------
model_keep_cols = [
    "EFFECTIVE_DATE",
    "FORECAST_DATE",
    "BRANCH_CODE",
    "TARGET_T_PLUS_1"
] + numeric_feature_cols + [
    "FORECAST_DAY_OF_WEEK",
    "FORECAST_MONTH"
]

# ป้องกันชื่อคอลัมน์ซ้ำ
model_keep_cols = list(dict.fromkeys(model_keep_cols))

model_data = (
    df_model[model_keep_cols]
    .copy()
    .sort_values(["FORECAST_DATE", "BRANCH_CODE"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5) ตรวจค่าที่หายและค่าที่ไม่เป็นตัวเลข
# ------------------------------------------------------------
missing_feature_count = (
    model_data[all_raw_feature_cols]
    .isna()
    .sum()
    .sum()
)

missing_target_count = (
    model_data["TARGET_T_PLUS_1"]
    .isna()
    .sum()
)

if missing_feature_count > 0:
    raise ValueError(
        f"ยังพบค่า Feature ที่หาย {missing_feature_count:,} ค่า"
    )

if missing_target_count > 0:
    raise ValueError(
        f"ยังพบค่า Target ที่หาย {missing_target_count:,} ค่า"
    )


# ------------------------------------------------------------
# 6) One-hot Encoding
# ------------------------------------------------------------
# drop_first=False เพราะต้องการเก็บตัวบ่งชี้แต่ละสาขาให้ครบ
# โมเดลต้นไม้ใช้ได้โดยไม่มีปัญหา
X = pd.get_dummies(
    model_data[all_raw_feature_cols],
    columns=categorical_feature_cols,
    drop_first=False,
    dtype=float
)

y = (
    model_data["TARGET_T_PLUS_1"]
    .astype(float)
    .copy()
)


# ------------------------------------------------------------
# 7) ตรวจค่าผิดปกติหลัง Encoding
# ------------------------------------------------------------
nan_in_X = X.isna().sum().sum()
infinite_in_X = np.isinf(X.to_numpy()).sum()

nan_in_y = y.isna().sum()
infinite_in_y = np.isinf(y.to_numpy()).sum()

duplicate_feature_names = X.columns.duplicated().sum()


# ------------------------------------------------------------
# 8) ตรวจป้องกัน Data Leakage
# ------------------------------------------------------------
forbidden_features = [
    "TARGET_T_PLUS_1",
    "TARGET_DIRECT_CHECK",
    "TARGET_IS_OBSERVED",
    "BASE_ROW_ELIGIBLE",
    "ACTUAL_NEXT_DATE",
    "EFFECTIVE_DATE",
    "FORECAST_DATE"
]

leaked_features = [
    col for col in forbidden_features
    if col in X.columns
]

if leaked_features:
    raise ValueError(
        f"พบตัวแปรที่ไม่ควรเข้าโมเดล: {leaked_features}"
    )


# Rolling30 ต้องอยู่ในชุด Feature
required_baseline = "TOTAL_CASH_OUT_ROLLING30"

if required_baseline not in X.columns:
    raise ValueError(
        "ไม่พบ TOTAL_CASH_OUT_ROLLING30 ในชุด Feature"
    )


# ------------------------------------------------------------
# 9) ตรวจจำนวนแถวให้ตรงกัน
# ------------------------------------------------------------
if not (
    len(model_data) == len(X) == len(y)
):
    raise ValueError(
        "จำนวนแถวของ model_data, X และ y ไม่ตรงกัน"
    )


# ------------------------------------------------------------
# 10) แสดงผลสรุป
# ------------------------------------------------------------
print("=" * 75)
print("จัดชุดข้อมูลสำหรับสร้างโมเดลสำเร็จ")
print("=" * 75)

print("จำนวนแถว Model-ready:", f"{len(model_data):,}")
print("จำนวน Features ก่อน Encoding:", len(all_raw_feature_cols))
print("จำนวน Features หลัง Encoding:", X.shape[1])

print(
    "ช่วงวันที่ข้อมูลวัน t:",
    model_data["EFFECTIVE_DATE"].min().date(),
    "ถึง",
    model_data["EFFECTIVE_DATE"].max().date()
)

print(
    "ช่วงวันที่พยากรณ์ t+1:",
    model_data["FORECAST_DATE"].min().date(),
    "ถึง",
    model_data["FORECAST_DATE"].max().date()
)

print("\nผลตรวจความสมบูรณ์:")
print("ค่า NaN ใน X:", nan_in_X)
print("ค่า Infinity ใน X:", infinite_in_X)
print("ค่า NaN ใน y:", nan_in_y)
print("ค่า Infinity ใน y:", infinite_in_y)
print("ชื่อ Feature ซ้ำ:", duplicate_feature_names)
print("ตัวแปร Leakage ที่พบ:", leaked_features)

print("\nสัดส่วน Target เท่ากับศูนย์:")
print(f"{(y == 0).mean() * 100:.2f}%")

print("\nรายชื่อ Features หลัง Encoding:")
for number, feature in enumerate(X.columns, start=1):
    print(f"{number:>3}. {feature}")

print("\nตัวอย่างข้อมูล Model-ready:")
display(
    model_data[
        [
            "EFFECTIVE_DATE",
            "FORECAST_DATE",
            "BRANCH_CODE",
            "TOTAL_CASH_OUT",
            "TOTAL_CASH_OUT_LAG1",
            "TOTAL_CASH_OUT_ROLLING30",
            "FORECAST_DAY_OF_WEEK",
            "FORECAST_IS_HOLIDAY",
            "TARGET_T_PLUS_1"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# STEP 10: Chronological Subtrain / Validation / Final Test split
# ============================================================

# ตรวจว่า index ของข้อมูลตรงกับ X และ y
if not model_data.index.equals(X.index):
    raise ValueError("Index ของ model_data และ X ไม่ตรงกัน")

if not model_data.index.equals(y.index):
    raise ValueError("Index ของ model_data และ y ไม่ตรงกัน")


# ------------------------------------------------------------
# 1) เรียงวันที่พยากรณ์ทั้งหมด
# ------------------------------------------------------------
unique_forecast_dates = np.array(
    sorted(model_data["FORECAST_DATE"].dropna().unique())
)

n_unique_dates = len(unique_forecast_dates)

if n_unique_dates < 30:
    raise ValueError("จำนวนวันที่น้อยเกินไปสำหรับแบ่งข้อมูล 3 ชุด")


# ------------------------------------------------------------
# 2) กำหนดสัดส่วนแบบ 64% / 16% / 20%
# ------------------------------------------------------------
# Development set = 80% แรก
# Final Test = 20% หลังสุด
development_end_index = int(np.floor(n_unique_dates * 0.80))

# ภายใน Development:
# Subtrain = 80% ของ Development
# Validation = 20% ของ Development
subtrain_end_index = int(
    np.floor(development_end_index * 0.80)
)

subtrain_dates = unique_forecast_dates[:subtrain_end_index]

validation_dates = unique_forecast_dates[
    subtrain_end_index:development_end_index
]

test_dates = unique_forecast_dates[
    development_end_index:
]


# ------------------------------------------------------------
# 3) สร้าง Mask จากวันที่พยากรณ์
# ------------------------------------------------------------
subtrain_mask = model_data["FORECAST_DATE"].isin(subtrain_dates)
validation_mask = model_data["FORECAST_DATE"].isin(validation_dates)
test_mask = model_data["FORECAST_DATE"].isin(test_dates)


# ------------------------------------------------------------
# 4) แยก Feature, Target และ Metadata
# ------------------------------------------------------------
X_subtrain = X.loc[subtrain_mask].copy()
y_subtrain = y.loc[subtrain_mask].copy()

X_validation = X.loc[validation_mask].copy()
y_validation = y.loc[validation_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


meta_cols = [
    "EFFECTIVE_DATE",
    "FORECAST_DATE",
    "BRANCH_CODE",
    "TARGET_T_PLUS_1"
]

meta_subtrain = model_data.loc[
    subtrain_mask, meta_cols
].copy()

meta_validation = model_data.loc[
    validation_mask, meta_cols
].copy()

meta_test = model_data.loc[
    test_mask, meta_cols
].copy()


# ------------------------------------------------------------
# 5) Development set สำหรับ Retrain หลังเลือกโมเดลเสร็จ
# ------------------------------------------------------------
development_mask = subtrain_mask | validation_mask

X_development = X.loc[development_mask].copy()
y_development = y.loc[development_mask].copy()

meta_development = model_data.loc[
    development_mask, meta_cols
].copy()


# ------------------------------------------------------------
# 6) Baseline Rolling30 สำหรับ Residual Model
# ------------------------------------------------------------
baseline_col = "TOTAL_CASH_OUT_ROLLING30"

baseline_subtrain = X_subtrain[baseline_col].to_numpy()
baseline_validation = X_validation[baseline_col].to_numpy()
baseline_test = X_test[baseline_col].to_numpy()
baseline_development = X_development[baseline_col].to_numpy()


# ------------------------------------------------------------
# 7) ตรวจสอบว่าไม่มีวันที่ทับซ้อนกัน
# ------------------------------------------------------------
subtrain_date_set = set(subtrain_dates)
validation_date_set = set(validation_dates)
test_date_set = set(test_dates)

overlap_subtrain_validation = (
    subtrain_date_set & validation_date_set
)

overlap_subtrain_test = (
    subtrain_date_set & test_date_set
)

overlap_validation_test = (
    validation_date_set & test_date_set
)

all_masks_count = (
    subtrain_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
)

unassigned_rows = (all_masks_count == 0).sum()
multiply_assigned_rows = (all_masks_count > 1).sum()


# ------------------------------------------------------------
# 8) ตรวจสอบลำดับเวลา
# ------------------------------------------------------------
chronological_order_valid = (
    max(subtrain_dates) < min(validation_dates)
    and max(validation_dates) < min(test_dates)
)

if overlap_subtrain_validation:
    raise ValueError("วันที่ Subtrain และ Validation ซ้ำกัน")

if overlap_subtrain_test:
    raise ValueError("วันที่ Subtrain และ Test ซ้ำกัน")

if overlap_validation_test:
    raise ValueError("วันที่ Validation และ Test ซ้ำกัน")

if unassigned_rows > 0:
    raise ValueError("มีบางแถวไม่ได้ถูกจัดเข้า Split")

if multiply_assigned_rows > 0:
    raise ValueError("มีบางแถวถูกจัดอยู่มากกว่าหนึ่ง Split")

if not chronological_order_valid:
    raise ValueError("ลำดับเวลาของ Split ไม่ถูกต้อง")


# ------------------------------------------------------------
# 9) ฟังก์ชันสรุปแต่ละ Split
# ------------------------------------------------------------
def summarize_split(name, meta, target):
    return {
        "Split": name,
        "Records": len(meta),
        "Unique_Forecast_Dates": meta["FORECAST_DATE"].nunique(),
        "Branches": meta["BRANCH_CODE"].nunique(),
        "Start_Forecast_Date": meta["FORECAST_DATE"].min(),
        "End_Forecast_Date": meta["FORECAST_DATE"].max(),
        "Target_Mean": target.mean(),
        "Target_Zero_Percent": (target == 0).mean() * 100
    }


split_summary = pd.DataFrame([
    summarize_split(
        "Subtrain",
        meta_subtrain,
        y_subtrain
    ),
    summarize_split(
        "Validation",
        meta_validation,
        y_validation
    ),
    summarize_split(
        "Final Test",
        meta_test,
        y_test
    )
])


# ------------------------------------------------------------
# 10) แสดงผล
# ------------------------------------------------------------
print("=" * 78)
print("แบ่งข้อมูลตามเวลาเรียบร้อย")
print("=" * 78)

print("จำนวนวันที่พยากรณ์ทั้งหมด:", f"{n_unique_dates:,}")
print("จำนวน Features:", X.shape[1])

print("\nผลตรวจ Split:")
print("วันที่ Subtrain/Validation ซ้ำ:", len(overlap_subtrain_validation))
print("วันที่ Subtrain/Test ซ้ำ:", len(overlap_subtrain_test))
print("วันที่ Validation/Test ซ้ำ:", len(overlap_validation_test))
print("แถวที่ไม่ถูกจัดเข้า Split:", unassigned_rows)
print("แถวที่อยู่มากกว่า 1 Split:", multiply_assigned_rows)
print("เรียงตามเวลาถูกต้อง:", chronological_order_valid)

print("\nสรุปแต่ละชุด:")
display(split_summary)

print("\nสัดส่วนจำนวน Records:")
print(
    pd.Series({
        "Subtrain (%)": len(X_subtrain) / len(X) * 100,
        "Validation (%)": len(X_validation) / len(X) * 100,
        "Final Test (%)": len(X_test) / len(X) * 100
    }).round(2)
)

print("\nช่วงวันที่:")
print(
    "Subtrain :",
    meta_subtrain["FORECAST_DATE"].min().date(),
    "ถึง",
    meta_subtrain["FORECAST_DATE"].max().date()
)

print(
    "Validation:",
    meta_validation["FORECAST_DATE"].min().date(),
    "ถึง",
    meta_validation["FORECAST_DATE"].max().date()
)

print(
    "Final Test:",
    meta_test["FORECAST_DATE"].min().date(),
    "ถึง",
    meta_test["FORECAST_DATE"].max().date()
)

print("\nสถานะ Final Test: ล็อกไว้ ยังไม่ใช้เลือกโมเดล")

In [ ]:
# ============================================================
# STEP 11: Evaluation functions and Validation baselines
# ============================================================

# ------------------------------------------------------------
# 1) ฟังก์ชันคำนวณ WAPE
# ------------------------------------------------------------
def calculate_wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.abs(y_true).sum()

    if denominator == 0:
        return np.nan

    return (
        np.abs(y_true - y_pred).sum()
        / denominator
        * 100
    )


# ------------------------------------------------------------
# 2) ฟังก์ชันคำนวณ Metrics
# ------------------------------------------------------------
def evaluate_predictions(model_name, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # ยอดเงินสดออกต้องไม่ติดลบ
    y_pred = np.clip(y_pred, 0, None)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    wape_value = calculate_wape(
        y_true,
        y_pred
    )

    # MAPE คำนวณเฉพาะวันที่ยอดจริงไม่เป็นศูนย์
    nonzero_mask = y_true != 0

    if nonzero_mask.any():
        mape_nonzero = (
            np.mean(
                np.abs(
                    (
                        y_true[nonzero_mask]
                        - y_pred[nonzero_mask]
                    )
                    / y_true[nonzero_mask]
                )
            )
            * 100
        )
    else:
        mape_nonzero = np.nan

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape_value,
        "MAPE_nonzero (%)": mape_nonzero
    }


# ------------------------------------------------------------
# 3) ตรวจ Baseline บน Validation เท่านั้น
# ------------------------------------------------------------
validation_baseline_results = []

# Baseline 1:
# ใช้ยอดเงินสดออกของวัน t ทำนายวัน t+1
naive_current_day_pred = (
    X_validation["TOTAL_CASH_OUT"]
    .to_numpy()
)

validation_baseline_results.append(
    evaluate_predictions(
        model_name="Naive: Current Day Cash-out",
        y_true=y_validation,
        y_pred=naive_current_day_pred
    )
)


# Baseline 2:
# ใช้ค่าเฉลี่ยย้อนหลัง 7 วันถึงวัน t
rolling7_validation_pred = (
    X_validation["TOTAL_CASH_OUT_ROLLING7"]
    .to_numpy()
)

validation_baseline_results.append(
    evaluate_predictions(
        model_name="Rolling Mean 7 Days",
        y_true=y_validation,
        y_pred=rolling7_validation_pred
    )
)


# Baseline 3:
# ใช้ค่าเฉลี่ยย้อนหลัง 30 วันถึงวัน t
rolling30_validation_pred = (
    X_validation["TOTAL_CASH_OUT_ROLLING30"]
    .to_numpy()
)

validation_baseline_results.append(
    evaluate_predictions(
        model_name="Rolling Mean 30 Days",
        y_true=y_validation,
        y_pred=rolling30_validation_pred
    )
)


# ------------------------------------------------------------
# 4) สร้างตารางเปรียบเทียบ Baseline
# ------------------------------------------------------------
validation_baseline_results = (
    pd.DataFrame(validation_baseline_results)
    .sort_values(
        ["WAPE (%)", "MAE"],
        ascending=True
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5) ตรวจความถูกต้อง
# ------------------------------------------------------------
validation_pred_lengths = {
    "Actual": len(y_validation),
    "Naive_Current_Day": len(naive_current_day_pred),
    "Rolling7": len(rolling7_validation_pred),
    "Rolling30": len(rolling30_validation_pred)
}

if len(set(validation_pred_lengths.values())) != 1:
    raise ValueError(
        "จำนวนแถวของ Validation และ Baseline Prediction ไม่ตรงกัน"
    )

if np.isnan(
    validation_baseline_results[
        ["MAE", "RMSE", "WAPE (%)"]
    ].to_numpy()
).any():
    raise ValueError(
        "พบค่า NaN ในผลการประเมิน Baseline"
    )


# ------------------------------------------------------------
# 6) แสดงผล
# ------------------------------------------------------------
print("=" * 76)
print("ประเมิน Baseline บน Validation สำเร็จ")
print("=" * 76)

print("จำนวน Validation records:", f"{len(y_validation):,}")
print(
    "ช่วงวันที่ Validation:",
    meta_validation["FORECAST_DATE"].min().date(),
    "ถึง",
    meta_validation["FORECAST_DATE"].max().date()
)

print("\nจำนวนข้อมูลที่ใช้คำนวณ:")
print(validation_pred_lengths)

print("\nผล Baseline บน Validation:")
display(
    validation_baseline_results.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "MAPE_nonzero (%)": "{:.2f}"
    })
)

best_baseline_name = (
    validation_baseline_results
    .iloc[0]["Model"]
)

best_baseline_wape = (
    validation_baseline_results
    .iloc[0]["WAPE (%)"]
)

print(
    "\nBaseline ที่ดีที่สุดบน Validation:",
    best_baseline_name
)

print(
    "Validation WAPE:",
    f"{best_baseline_wape:.2f}%"
)

print("\nสถานะ Final Test: ยังไม่ได้ถูกเรียกใช้")

In [ ]:
# ============================================================
# Add Seasonal Naive baseline to Validation results
# Uses actual TOTAL_CASH_OUT from forecast date - 7 days
# ============================================================

seasonal_lookup = (
    df_features[
        ["BRANCH_CODE", "EFFECTIVE_DATE", "TOTAL_CASH_OUT"]
    ]
    .rename(columns={
        "EFFECTIVE_DATE": "SEASONAL_DATE",
        "TOTAL_CASH_OUT": "SEASONAL_NAIVE_7"
    })
)

seasonal_validation = meta_validation[
    ["BRANCH_CODE", "FORECAST_DATE"]
].copy()

seasonal_validation["_ROW_ORDER"] = np.arange(
    len(seasonal_validation)
)

seasonal_validation["SEASONAL_DATE"] = (
    seasonal_validation["FORECAST_DATE"]
    - pd.Timedelta(days=7)
)

seasonal_validation = (
    seasonal_validation
    .merge(
        seasonal_lookup,
        on=["BRANCH_CODE", "SEASONAL_DATE"],
        how="left",
        validate="many_to_one"
    )
    .sort_values("_ROW_ORDER")
)

if seasonal_validation["SEASONAL_NAIVE_7"].isna().any():
    raise ValueError(
        "Seasonal Naive มีค่าหาย ต้องตรวจสอบวันที่ย้อนหลัง 7 วัน"
    )

seasonal7_validation_pred = (
    seasonal_validation["SEASONAL_NAIVE_7"].to_numpy()
)

seasonal7_result = evaluate_predictions(
    model_name="Seasonal Naive: Same Weekday",
    y_true=y_validation,
    y_pred=seasonal7_validation_pred
)

# ป้องกันแถวซ้ำหากรันเซลล์นี้มากกว่าหนึ่งครั้ง
validation_baseline_results = validation_baseline_results[
    validation_baseline_results["Model"]
    != "Seasonal Naive: Same Weekday"
]

validation_baseline_results = (
    pd.concat(
        [
            validation_baseline_results,
            pd.DataFrame([seasonal7_result])
        ],
        ignore_index=True
    )
    .sort_values(["WAPE (%)", "MAE"])
    .reset_index(drop=True)
)

display(
    validation_baseline_results.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "MAPE_nonzero (%)": "{:.2f}"
    })
)

In [ ]:
# ============================================================
# STEP 12: Compare ML model families on Validation
# ============================================================

import time

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1) กำหนดขนาดโมเดล
# ------------------------------------------------------------
# FAST_MODE=False ใช้ค่าหลักสำหรับเก็บผล Final
N_TREES = 150 if FAST_MODE else 500
HGB_ITERATIONS = 150 if FAST_MODE else 500
XGB_TREES = 200 if FAST_MODE else 500


# ------------------------------------------------------------
# 2) กำหนดโมเดลที่ต้องการเปรียบเทียบ
# ------------------------------------------------------------
candidate_models = {

    # Ridge ใช้แทน Linear Regression ธรรมดา
    # เพราะรองรับปัญหาตัวแปรสัมพันธ์กันสูงได้ดีกว่า
    "Ridge Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            Ridge(alpha=1.0)
        )
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=N_TREES,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=N_TREES,
        max_depth=None,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.05,
        max_iter=HGB_ITERATIONS,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.1,
        early_stopping=False,
        random_state=RANDOM_STATE
    ),

    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        eval_metric="mae",
        n_estimators=XGB_TREES,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    )
}


# ------------------------------------------------------------
# 3) เตรียมที่เก็บผลลัพธ์
# ------------------------------------------------------------
fitted_candidate_models = {}
validation_ml_predictions = {}
validation_ml_results = []


# ------------------------------------------------------------
# 4) Train ด้วย Subtrain และประเมิน Validation
# ------------------------------------------------------------
for model_name, model in candidate_models.items():

    print("=" * 70)
    print("กำลัง Train:", model_name)

    start_time = time.time()

    model.fit(
        X_subtrain,
        y_subtrain
    )

    raw_prediction = model.predict(
        X_validation
    )

    # เงินสดออกไม่สามารถติดลบ
    validation_prediction = np.clip(
        raw_prediction,
        0,
        None
    )

    elapsed_seconds = time.time() - start_time

    result = evaluate_predictions(
        model_name=model_name,
        y_true=y_validation,
        y_pred=validation_prediction
    )

    result["Training_Time_Seconds"] = elapsed_seconds

    validation_ml_results.append(result)

    fitted_candidate_models[model_name] = model
    validation_ml_predictions[model_name] = (
        validation_prediction
    )

    print(
        f"เสร็จแล้ว | "
        f"WAPE = {result['WAPE (%)']:.2f}% | "
        f"เวลา = {elapsed_seconds:.2f} วินาที"
    )


# ------------------------------------------------------------
# 5) สร้างตารางผล ML
# ------------------------------------------------------------
validation_ml_results = (
    pd.DataFrame(validation_ml_results)
    .sort_values(
        ["WAPE (%)", "MAE"],
        ascending=True
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6) รวมผล Baseline และ ML เพื่อเปรียบเทียบ
# ------------------------------------------------------------
validation_all_results = pd.concat(
    [
        validation_baseline_results.assign(
            Training_Time_Seconds=0.0,
            Model_Type="Baseline"
        ),
        validation_ml_results.assign(
            Model_Type="Machine Learning"
        )
    ],
    ignore_index=True
)

validation_all_results = (
    validation_all_results
    .sort_values(
        ["WAPE (%)", "MAE"],
        ascending=True
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7) ตรวจความถูกต้อง
# ------------------------------------------------------------
for model_name, prediction in validation_ml_predictions.items():

    if len(prediction) != len(y_validation):
        raise ValueError(
            f"Prediction ของ {model_name} มีจำนวนแถวไม่ตรง"
        )

    if np.isnan(prediction).any():
        raise ValueError(
            f"Prediction ของ {model_name} มีค่า NaN"
        )

    if np.isinf(prediction).any():
        raise ValueError(
            f"Prediction ของ {model_name} มีค่า Infinity"
        )

    if (prediction < 0).any():
        raise ValueError(
            f"Prediction ของ {model_name} ยังมีค่าติดลบ"
        )


# ------------------------------------------------------------
# 8) หาโมเดล ML ที่ดีที่สุดบน Validation
# ------------------------------------------------------------
best_absolute_model_name = (
    validation_ml_results.iloc[0]["Model"]
)

best_absolute_validation_wape = (
    validation_ml_results.iloc[0]["WAPE (%)"]
)

best_absolute_model = (
    fitted_candidate_models[best_absolute_model_name]
)


# ------------------------------------------------------------
# 9) แสดงผล
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("เปรียบเทียบโมเดลบน Validation สำเร็จ")
print("=" * 78)

print("\nผล Machine Learning:")
display(
    validation_ml_results.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "MAPE_nonzero (%)": "{:.2f}",
        "Training_Time_Seconds": "{:,.2f}"
    })
)

print("\nเปรียบเทียบ Baseline และ Machine Learning:")
display(
    validation_all_results[
        [
            "Model",
            "Model_Type",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Training_Time_Seconds"
        ]
    ].style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "Training_Time_Seconds": "{:,.2f}"
    })
)

print(
    "\nโมเดล ML แบบทำนายยอดโดยตรงที่ดีที่สุดบน Validation:",
    best_absolute_model_name
)

print(
    "Validation WAPE:",
    f"{best_absolute_validation_wape:.2f}%"
)

print(
    "Baseline ที่ดีที่สุด:",
    best_baseline_name,
    f"({best_baseline_wape:.2f}%)"
)

improvement_vs_baseline = (
    best_baseline_wape
    - best_absolute_validation_wape
)

print(
    "WAPE ลดลงจาก Baseline:",
    f"{improvement_vs_baseline:.2f} percentage points"
)

print("\nสถานะ Final Test: ยังไม่ได้ถูกเรียกใช้")

In [ ]:
# ============================================================
# STEP 13: Tune HGB residual models on Validation only
# ============================================================

import time

# ------------------------------------------------------------
# 1) Baseline ที่จะทดลองกับ Residual Model
# ------------------------------------------------------------
residual_baselines = {
    "Rolling7": {
        "subtrain": X_subtrain[
            "TOTAL_CASH_OUT_ROLLING7"
        ].to_numpy(),

        "validation": X_validation[
            "TOTAL_CASH_OUT_ROLLING7"
        ].to_numpy()
    },

    "Rolling30": {
        "subtrain": X_subtrain[
            "TOTAL_CASH_OUT_ROLLING30"
        ].to_numpy(),

        "validation": X_validation[
            "TOTAL_CASH_OUT_ROLLING30"
        ].to_numpy()
    }
}


# ------------------------------------------------------------
# 2) ระดับ Clipping ที่จะทดลอง
# None = ไม่ทำ Residual Clipping
# ตัวเลขอื่นคือ Quantile ที่ตัดจากหัวและท้าย
# ------------------------------------------------------------
clip_quantiles = [
    None,
    0.005,
    0.01,
    0.02,
    0.05
]


# ------------------------------------------------------------
# 3) เตรียมตัวแปรเก็บผล
# ------------------------------------------------------------
residual_candidate_models = {}
residual_candidate_info = {}
residual_validation_predictions = {}
residual_validation_results = []


# ------------------------------------------------------------
# 4) ทดลองแต่ละ Baseline และแต่ละระดับ Clipping
# ------------------------------------------------------------
for baseline_name, baseline_values in residual_baselines.items():

    baseline_subtrain_values = baseline_values["subtrain"]
    baseline_validation_values = baseline_values["validation"]

    # Residual จริงใน Subtrain
    raw_residual_subtrain = (
        y_subtrain.to_numpy()
        - baseline_subtrain_values
    )

    for clip_quantile in clip_quantiles:

        # ----------------------------------------------
        # กำหนดขอบเขต Residual จาก Subtrain เท่านั้น
        # ----------------------------------------------
        if clip_quantile is None:

            lower_bound = None
            upper_bound = None

            residual_target_for_training = (
                raw_residual_subtrain.copy()
            )

            clip_label = "NoClip"

        else:

            lower_bound = np.quantile(
                raw_residual_subtrain,
                clip_quantile
            )

            upper_bound = np.quantile(
                raw_residual_subtrain,
                1 - clip_quantile
            )

            residual_target_for_training = np.clip(
                raw_residual_subtrain,
                lower_bound,
                upper_bound
            )

            clip_label = (
                f"Clip_{clip_quantile * 100:.1f}pct"
            )


        candidate_id = (
            f"HGB_Residual_{baseline_name}_{clip_label}"
        )

        print("=" * 75)
        print("กำลัง Train:", candidate_id)

        start_time = time.time()


        # ----------------------------------------------
        # สร้าง HGB Model ใหม่ทุก Candidate
        # ----------------------------------------------
        residual_model = HistGradientBoostingRegressor(
            loss="absolute_error",
            learning_rate=0.05,
            max_iter=HGB_ITERATIONS,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.1,
            early_stopping=False,
            random_state=RANDOM_STATE
        )


        # ----------------------------------------------
        # Train ด้วย Subtrain Residual
        # ----------------------------------------------
        residual_model.fit(
            X_subtrain,
            residual_target_for_training
        )


        # ----------------------------------------------
        # พยากรณ์ Residual บน Validation
        # ----------------------------------------------
        predicted_residual = residual_model.predict(
            X_validation
        )


        # ทำ Prediction Clipping ด้วยขอบเขตเดียวกับ Training
        if clip_quantile is not None:

            predicted_residual = np.clip(
                predicted_residual,
                lower_bound,
                upper_bound
            )


        # ----------------------------------------------
        # Final Prediction = Baseline + Residual
        # ----------------------------------------------
        validation_prediction = np.clip(
            baseline_validation_values
            + predicted_residual,
            0,
            None
        )

        elapsed_seconds = time.time() - start_time


        # ----------------------------------------------
        # ประเมิน Validation
        # ----------------------------------------------
        result = evaluate_predictions(
            model_name=candidate_id,
            y_true=y_validation,
            y_pred=validation_prediction
        )

        result["Baseline"] = baseline_name

        result["Clip_Quantile"] = (
            np.nan
            if clip_quantile is None
            else clip_quantile
        )

        result["Residual_Lower_Bound"] = (
            np.nan
            if lower_bound is None
            else lower_bound
        )

        result["Residual_Upper_Bound"] = (
            np.nan
            if upper_bound is None
            else upper_bound
        )

        result["Training_Time_Seconds"] = (
            elapsed_seconds
        )


        # ----------------------------------------------
        # เก็บผลและตัวโมเดล
        # ----------------------------------------------
        residual_validation_results.append(result)

        residual_candidate_models[candidate_id] = (
            residual_model
        )

        residual_validation_predictions[candidate_id] = (
            validation_prediction
        )

        residual_candidate_info[candidate_id] = {
            "Baseline": baseline_name,
            "Clip_Quantile": clip_quantile,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound
        }

        print(
            f"เสร็จแล้ว | "
            f"WAPE = {result['WAPE (%)']:.2f}% | "
            f"เวลา = {elapsed_seconds:.2f} วินาที"
        )


# ------------------------------------------------------------
# 5) สร้างตารางผล Residual Model
# ------------------------------------------------------------
residual_validation_results = (
    pd.DataFrame(residual_validation_results)
    .sort_values(
        ["WAPE (%)", "MAE"],
        ascending=True
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6) ตรวจสอบ Prediction
# ------------------------------------------------------------
for candidate_id, prediction in (
    residual_validation_predictions.items()
):

    if len(prediction) != len(y_validation):
        raise ValueError(
            f"จำนวน Prediction ของ {candidate_id} ไม่ตรง"
        )

    if np.isnan(prediction).any():
        raise ValueError(
            f"Prediction ของ {candidate_id} มี NaN"
        )

    if np.isinf(prediction).any():
        raise ValueError(
            f"Prediction ของ {candidate_id} มี Infinity"
        )

    if (prediction < 0).any():
        raise ValueError(
            f"Prediction ของ {candidate_id} ติดลบ"
        )


# ------------------------------------------------------------
# 7) หา Residual Candidate ที่ดีที่สุด
# ------------------------------------------------------------
best_residual_candidate_id = (
    residual_validation_results.iloc[0]["Model"]
)

best_residual_validation_wape = (
    residual_validation_results.iloc[0]["WAPE (%)"]
)

best_residual_model = (
    residual_candidate_models[
        best_residual_candidate_id
    ]
)

best_residual_info = (
    residual_candidate_info[
        best_residual_candidate_id
    ]
)

best_residual_validation_prediction = (
    residual_validation_predictions[
        best_residual_candidate_id
    ]
)


# ------------------------------------------------------------
# 8) ดึงผล HGB แบบทำนายยอดโดยตรงมาเปรียบเทียบ
# ------------------------------------------------------------
direct_hgb_result = (
    validation_ml_results.loc[
        validation_ml_results["Model"]
        == "HistGradientBoosting"
    ]
    .iloc[0]
)

direct_hgb_validation_wape = (
    direct_hgb_result["WAPE (%)"]
)


# ------------------------------------------------------------
# 9) ตารางสรุปเปรียบเทียบ
# ------------------------------------------------------------
residual_vs_direct_summary = pd.DataFrame([
    {
        "Approach": "Best Baseline",
        "Model": best_baseline_name,
        "Validation WAPE (%)": best_baseline_wape
    },
    {
        "Approach": "Direct ML",
        "Model": "HistGradientBoosting",
        "Validation WAPE (%)": direct_hgb_validation_wape
    },
    {
        "Approach": "Residual ML",
        "Model": best_residual_candidate_id,
        "Validation WAPE (%)": (
            best_residual_validation_wape
        )
    }
]).sort_values(
    "Validation WAPE (%)",
    ascending=True
).reset_index(drop=True)


# ------------------------------------------------------------
# 10) แสดงผล
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("ทดลอง HGB Residual บน Validation สำเร็จ")
print("=" * 80)

print("\nผล Residual Candidates:")
display(
    residual_validation_results[
        [
            "Model",
            "Baseline",
            "Clip_Quantile",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Residual_Lower_Bound",
            "Residual_Upper_Bound",
            "Training_Time_Seconds"
        ]
    ].style.format({
        "Clip_Quantile": "{:.3f}",
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "Residual_Lower_Bound": "{:,.2f}",
        "Residual_Upper_Bound": "{:,.2f}",
        "Training_Time_Seconds": "{:,.2f}"
    })
)

print("\nเปรียบเทียบแนวทางหลัก:")
display(
    residual_vs_direct_summary.style.format({
        "Validation WAPE (%)": "{:.2f}"
    })
)

print(
    "\nResidual Model ที่ดีที่สุด:",
    best_residual_candidate_id
)

print(
    "Baseline:",
    best_residual_info["Baseline"]
)

print(
    "Clip Quantile:",
    best_residual_info["Clip_Quantile"]
)

print(
    "Validation WAPE:",
    f"{best_residual_validation_wape:.2f}%"
)

print(
    "ผลต่างเมื่อเทียบกับ Direct HGB:",
    f"{direct_hgb_validation_wape - best_residual_validation_wape:.2f}",
    "percentage points"
)

print("\nสถานะ Final Test: ยังไม่ได้ถูกเรียกใช้")

In [ ]:
# ============================================================
# STEP 14: Retrain locked final model and evaluate Final Test once
# ============================================================

import time

# ------------------------------------------------------------
# 1) ล็อกโครงสร้างที่ชนะจาก Validation
# ห้ามเปลี่ยนหลังเห็นผล Final Test
# ------------------------------------------------------------
LOCKED_BASELINE = "Rolling7"
LOCKED_CLIP_QUANTILE = 0.01

LOCKED_MODEL_NAME = (
    "HGB Residual Rolling7 + Residual Clipping 1%"
)

print("=" * 80)
print("Locked Final Model")
print("=" * 80)
print("Model:", LOCKED_MODEL_NAME)
print("Baseline:", LOCKED_BASELINE)
print("Clip Quantile:", LOCKED_CLIP_QUANTILE)
print(
    "เหตุผลที่เลือก: Validation WAPE ต่ำที่สุด",
    f"({best_residual_validation_wape:.2f}%)"
)


# ------------------------------------------------------------
# 2) เตรียม Rolling7 Baseline
# ------------------------------------------------------------
final_baseline_col = "TOTAL_CASH_OUT_ROLLING7"

development_baseline = (
    X_development[final_baseline_col]
    .to_numpy()
)

test_baseline = (
    X_test[final_baseline_col]
    .to_numpy()
)


# ------------------------------------------------------------
# 3) สร้าง Residual Target จาก Development เท่านั้น
# ------------------------------------------------------------
development_residual_raw = (
    y_development.to_numpy()
    - development_baseline
)


# ------------------------------------------------------------
# 4) คำนวณขอบเขต Clipping ใหม่จาก Development เท่านั้น
# ------------------------------------------------------------
final_residual_lower_bound = np.quantile(
    development_residual_raw,
    LOCKED_CLIP_QUANTILE
)

final_residual_upper_bound = np.quantile(
    development_residual_raw,
    1 - LOCKED_CLIP_QUANTILE
)

development_residual_clipped = np.clip(
    development_residual_raw,
    final_residual_lower_bound,
    final_residual_upper_bound
)


# ------------------------------------------------------------
# 5) สร้าง Final HGB Model ใหม่
# ------------------------------------------------------------
final_model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.05,
    max_iter=HGB_ITERATIONS,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.1,
    early_stopping=False,
    random_state=RANDOM_STATE
)


# ------------------------------------------------------------
# 6) Retrain ด้วย Development 80%
# ------------------------------------------------------------
start_time = time.time()

final_model.fit(
    X_development,
    development_residual_clipped
)

final_training_time = time.time() - start_time


# ------------------------------------------------------------
# 7) เปิด Final Test และพยากรณ์เพียงครั้งเดียว
# ------------------------------------------------------------
test_residual_prediction_raw = final_model.predict(
    X_test
)

# ใช้ขอบเขตที่คำนวณจาก Development เท่านั้น
test_residual_prediction_clipped = np.clip(
    test_residual_prediction_raw,
    final_residual_lower_bound,
    final_residual_upper_bound
)

# Final prediction = Rolling7 baseline + predicted residual
final_test_prediction = np.clip(
    test_baseline
    + test_residual_prediction_clipped,
    0,
    None
)


# ------------------------------------------------------------
# 8) ประเมิน Final Test
# ------------------------------------------------------------
final_test_result = evaluate_predictions(
    model_name=LOCKED_MODEL_NAME,
    y_true=y_test,
    y_pred=final_test_prediction
)

final_test_result["Training_Time_Seconds"] = (
    final_training_time
)

final_test_results = pd.DataFrame([
    final_test_result
])


# ------------------------------------------------------------
# 9) ประเมิน Rolling7 Baseline บน Final Test เพื่อใช้อ้างอิง
# ------------------------------------------------------------
final_test_baseline_result = evaluate_predictions(
    model_name="Rolling Mean 7 Days",
    y_true=y_test,
    y_pred=test_baseline
)

final_test_comparison = pd.DataFrame([
    final_test_result,
    final_test_baseline_result
]).sort_values(
    ["WAPE (%)", "MAE"],
    ascending=True
).reset_index(drop=True)


# ------------------------------------------------------------
# 10) สร้างตาราง Prediction ราย Branch-day
# ------------------------------------------------------------
final_pred_df = (
    meta_test[
        [
            "EFFECTIVE_DATE",
            "FORECAST_DATE",
            "BRANCH_CODE",
            "TARGET_T_PLUS_1"
        ]
    ]
    .reset_index(drop=True)
    .copy()
)

final_pred_df["ROLLING7_BASELINE"] = (
    test_baseline
)

final_pred_df["PREDICTED_RESIDUAL_RAW"] = (
    test_residual_prediction_raw
)

final_pred_df["PREDICTED_RESIDUAL_CLIPPED"] = (
    test_residual_prediction_clipped
)

final_pred_df["PRED_FINAL"] = (
    final_test_prediction
)

final_pred_df["ABSOLUTE_ERROR"] = np.abs(
    final_pred_df["TARGET_T_PLUS_1"]
    - final_pred_df["PRED_FINAL"]
)

final_pred_df["ERROR"] = (
    final_pred_df["PRED_FINAL"]
    - final_pred_df["TARGET_T_PLUS_1"]
)


# ------------------------------------------------------------
# 11) ตรวจความถูกต้อง
# ------------------------------------------------------------
if len(final_pred_df) != len(y_test):
    raise ValueError(
        "จำนวนแถว Final Prediction ไม่ตรงกับ Final Test"
    )

if not np.allclose(
    final_pred_df["TARGET_T_PLUS_1"].to_numpy(),
    y_test.to_numpy()
):
    raise ValueError(
        "Target ใน final_pred_df ไม่ตรงกับ y_test"
    )

if not np.allclose(
    final_pred_df["PRED_FINAL"].to_numpy(),
    final_test_prediction
):
    raise ValueError(
        "PRED_FINAL ไม่ตรงกับ Final Prediction"
    )

if np.isnan(final_test_prediction).any():
    raise ValueError(
        "Final Prediction มีค่า NaN"
    )

if np.isinf(final_test_prediction).any():
    raise ValueError(
        "Final Prediction มีค่า Infinity"
    )

if (final_test_prediction < 0).any():
    raise ValueError(
        "Final Prediction มีค่าติดลบ"
    )


# ------------------------------------------------------------
# 12) คำนวณผลต่างจาก Baseline
# ------------------------------------------------------------
final_model_wape = final_test_result["WAPE (%)"]

final_baseline_wape = (
    final_test_baseline_result["WAPE (%)"]
)

final_wape_improvement = (
    final_baseline_wape
    - final_model_wape
)


# ------------------------------------------------------------
# 13) บันทึกผล
# ------------------------------------------------------------
final_pred_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_test_predictions.csv"
    ),
    index=False
)

final_test_comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_test_results.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 14) แสดงผล
# ------------------------------------------------------------
print("\n" + "=" * 82)
print("ประเมิน Final Test สำเร็จ")
print("=" * 82)

print(
    "ช่วงวันที่ Final Test:",
    final_pred_df["FORECAST_DATE"].min().date(),
    "ถึง",
    final_pred_df["FORECAST_DATE"].max().date()
)

print(
    "จำนวน Final Test records:",
    f"{len(final_pred_df):,}"
)

print(
    "จำนวนสาขา:",
    final_pred_df["BRANCH_CODE"].nunique()
)

print("\nResidual Clipping Bounds จาก Development:")
print(
    "Lower Bound:",
    f"{final_residual_lower_bound:,.2f}"
)
print(
    "Upper Bound:",
    f"{final_residual_upper_bound:,.2f}"
)

print("\nผล Final Test:")
display(
    final_test_comparison[
        [
            "Model",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "MAPE_nonzero (%)"
        ]
    ].style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "MAPE_nonzero (%)": "{:.2f}"
    })
)

print(
    "\nFinal Model WAPE:",
    f"{final_model_wape:.2f}%"
)

print(
    "Rolling7 Baseline WAPE:",
    f"{final_baseline_wape:.2f}%"
)

print(
    "Final Model ลด WAPE จาก Baseline:",
    f"{final_wape_improvement:.2f}",
    "percentage points"
)

print("\nตัวอย่าง Final Predictions:")
display(final_pred_df.head(15))

print("\nบันทึกไฟล์แล้วที่:")
print(
    os.path.join(
        OUTPUT_DIR,
        "final_test_predictions.csv"
    )
)
print(
    os.path.join(
        OUTPUT_DIR,
        "final_test_results.csv"
    )
)

print("\nสถานะ: Final Test ถูกใช้ประเมินแล้ว ห้ามย้อนกลับไปปรับโมเดล")

In [ ]:
# ============================================================
# STEP 15: Final Test performance by branch
# ============================================================

def branch_metrics(group):
    actual = group["TARGET_T_PLUS_1"].to_numpy(dtype=float)
    predicted = group["PRED_FINAL"].to_numpy(dtype=float)

    absolute_error = np.abs(actual - predicted)
    denominator = np.abs(actual).sum()

    branch_wape = (
        absolute_error.sum() / denominator * 100
        if denominator > 0
        else np.nan
    )

    branch_mae = mean_absolute_error(
        actual,
        predicted
    )

    branch_rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    # Prediction > Actual หมายถึงเตรียมเงินเกิน
    over_prediction_mask = predicted > actual

    # Prediction < Actual หมายถึงเตรียมเงินไม่พอ
    under_prediction_mask = predicted < actual

    return pd.Series({
        "Records": len(group),
        "Actual_Total": actual.sum(),
        "Predicted_Total": predicted.sum(),
        "Absolute_Error_Total": absolute_error.sum(),
        "MAE": branch_mae,
        "RMSE": branch_rmse,
        "WAPE (%)": branch_wape,
        "Actual_Mean": actual.mean(),
        "Predicted_Mean": predicted.mean(),
        "Zero_Actual_Percent": (
            (actual == 0).mean() * 100
        ),
        "Underprediction_Records": (
            under_prediction_mask.sum()
        ),
        "Underprediction_Percent": (
            under_prediction_mask.mean() * 100
        ),
        "Overprediction_Records": (
            over_prediction_mask.sum()
        ),
        "Overprediction_Percent": (
            over_prediction_mask.mean() * 100
        )
    })


# ------------------------------------------------------------
# 1) คำนวณผลรายสาขา
# ------------------------------------------------------------
branch_performance = (
    final_pred_df
    .groupby("BRANCH_CODE", sort=True)
    .apply(branch_metrics)
    .reset_index()
    .sort_values(
        "WAPE (%)",
        ascending=True
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2) ตรวจว่าแต่ละสาขามีจำนวนวันเท่ากันหรือไม่
# ------------------------------------------------------------
branch_record_counts = (
    final_pred_df
    .groupby("BRANCH_CODE")
    .size()
)

record_count_consistent = (
    branch_record_counts.nunique() == 1
)


# ------------------------------------------------------------
# 3) ตรวจว่าเมื่อนำผลรายสาขามารวมแล้ว
# ต้องกลับมาได้ Overall WAPE เดิม
# ------------------------------------------------------------
reconstructed_overall_wape = (
    branch_performance[
        "Absolute_Error_Total"
    ].sum()
    / branch_performance[
        "Actual_Total"
    ].abs().sum()
    * 100
)

overall_wape_difference = abs(
    reconstructed_overall_wape
    - final_model_wape
)

if overall_wape_difference > 1e-8:
    raise ValueError(
        "WAPE จากผลรายสาขาไม่ตรงกับ Overall WAPE"
    )


# ------------------------------------------------------------
# 4) เพิ่มอันดับ
# ------------------------------------------------------------
branch_performance.insert(
    1,
    "WAPE_Rank",
    np.arange(
        1,
        len(branch_performance) + 1
    )
)


# ------------------------------------------------------------
# 5) บันทึกผล
# ------------------------------------------------------------
branch_performance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_test_branch_performance.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 6) แสดงผล
# ------------------------------------------------------------
print("=" * 82)
print("ประเมิน Final Test แยกรายสาขาสำเร็จ")
print("=" * 82)

print(
    "จำนวนสาขา:",
    branch_performance["BRANCH_CODE"].nunique()
)

print(
    "จำนวน Records ต่อสาขา:",
    sorted(branch_record_counts.unique())
)

print(
    "ทุกสาขามีจำนวน Records เท่ากัน:",
    record_count_consistent
)

print(
    "Overall WAPE จากการรวมผลรายสาขา:",
    f"{reconstructed_overall_wape:.2f}%"
)

print(
    "Overall WAPE ของ Final Model:",
    f"{final_model_wape:.2f}%"
)

print(
    "ผลต่าง:",
    f"{overall_wape_difference:.10f}"
)

print("\nผลรายสาขา เรียงจาก WAPE ต่ำไปสูง:")

display(
    branch_performance[
        [
            "WAPE_Rank",
            "BRANCH_CODE",
            "Records",
            "Actual_Total",
            "Predicted_Total",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Zero_Actual_Percent",
            "Underprediction_Percent",
            "Overprediction_Percent"
        ]
    ].style.format({
        "Actual_Total": "{:,.2f}",
        "Predicted_Total": "{:,.2f}",
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "WAPE (%)": "{:.2f}",
        "Zero_Actual_Percent": "{:.2f}",
        "Underprediction_Percent": "{:.2f}",
        "Overprediction_Percent": "{:.2f}"
    })
)

best_branch = branch_performance.iloc[0]
worst_branch = branch_performance.iloc[-1]

print(
    "\nสาขาที่ WAPE ต่ำที่สุด:",
    best_branch["BRANCH_CODE"],
    f"({best_branch['WAPE (%)']:.2f}%)"
)

print(
    "สาขาที่ WAPE สูงที่สุด:",
    worst_branch["BRANCH_CODE"],
    f"({worst_branch['WAPE (%)']:.2f}%)"
)

print("\nบันทึกไฟล์แล้วที่:")
print(
    os.path.join(
        OUTPUT_DIR,
        "final_test_branch_performance.csv"
    )
)

In [ ]:
# ============================================================
# STEP 16: Permutation importance for the locked final pipeline
# ============================================================

# จำนวนครั้งที่สุ่มสลับค่าของแต่ละ Feature
N_PERMUTATION_REPEATS = 5

# ใช้ random seed เดิมเพื่อให้ผลทำซ้ำได้
rng = np.random.default_rng(RANDOM_STATE)


# ------------------------------------------------------------
# 1) ฟังก์ชันพยากรณ์ของ Final Pipeline
# ------------------------------------------------------------
def predict_final_pipeline(X_input):
    """
    Final prediction =
    Rolling7 baseline
    + HGB-predicted residual

    จากนั้น:
    1) Clip residual ด้วยขอบเขตที่คำนวณจาก Development
    2) Clip final prediction ไม่ให้ติดลบ
    """

    residual_raw = final_model.predict(X_input)

    residual_clipped = np.clip(
        residual_raw,
        final_residual_lower_bound,
        final_residual_upper_bound
    )

    baseline_values = (
        X_input[final_baseline_col]
        .to_numpy(dtype=float)
    )

    prediction = np.clip(
        baseline_values + residual_clipped,
        0,
        None
    )

    return prediction


# ------------------------------------------------------------
# 2) ยืนยันว่า Pipeline สร้าง Prediction ตรงกับผล Final เดิม
# ------------------------------------------------------------
pipeline_base_prediction = predict_final_pipeline(
    X_test
)

prediction_match = np.allclose(
    pipeline_base_prediction,
    final_test_prediction,
    rtol=1e-10,
    atol=1e-6
)

if not prediction_match:
    raise ValueError(
        "สูตร Final Pipeline ไม่ตรงกับ Final Prediction เดิม"
    )

base_test_wape = calculate_wape(
    y_test,
    pipeline_base_prediction
)

if not np.isclose(
    base_test_wape,
    final_model_wape,
    rtol=1e-10,
    atol=1e-8
):
    raise ValueError(
        "WAPE จาก Pipeline ไม่ตรงกับ Final Model WAPE"
    )


# ------------------------------------------------------------
# 3) Permutation Importance
# ------------------------------------------------------------
permutation_rows = []

print("=" * 82)
print("เริ่มคำนวณ Permutation Importance")
print("=" * 82)
print("จำนวน Features:", X_test.shape[1])
print("จำนวนครั้งที่สลับต่อ Feature:", N_PERMUTATION_REPEATS)
print("Base Final Test WAPE:", f"{base_test_wape:.2f}%")
print()


for feature_number, feature_name in enumerate(
    X_test.columns,
    start=1
):

    wape_increases = []

    original_values = (
        X_test[feature_name]
        .to_numpy(copy=True)
    )

    for repeat in range(N_PERMUTATION_REPEATS):

        X_permuted = X_test.copy()

        X_permuted[feature_name] = rng.permutation(
            original_values
        )

        permuted_prediction = predict_final_pipeline(
            X_permuted
        )

        permuted_wape = calculate_wape(
            y_test,
            permuted_prediction
        )

        wape_increase = (
            permuted_wape - base_test_wape
        )

        wape_increases.append(wape_increase)


    permutation_rows.append({
        "Feature": feature_name,
        "Importance_Mean_WAPE_Increase": np.mean(
            wape_increases
        ),
        "Importance_Std": np.std(
            wape_increases,
            ddof=0
        ),
        "Min_WAPE_Increase": np.min(
            wape_increases
        ),
        "Max_WAPE_Increase": np.max(
            wape_increases
        )
    })

    print(
        f"{feature_number:>3}/{X_test.shape[1]} "
        f"{feature_name:<45} "
        f"Increase = {np.mean(wape_increases):>8.4f}"
    )


# ------------------------------------------------------------
# 4) สร้างตารางผล
# ------------------------------------------------------------
permutation_importance_df = (
    pd.DataFrame(permutation_rows)
    .sort_values(
        "Importance_Mean_WAPE_Increase",
        ascending=False
    )
    .reset_index(drop=True)
)

permutation_importance_df.insert(
    0,
    "Importance_Rank",
    np.arange(
        1,
        len(permutation_importance_df) + 1
    )
)


# ------------------------------------------------------------
# 5) บันทึกผล
# ------------------------------------------------------------
permutation_importance_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_pipeline_permutation_importance.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 6) สร้างกราฟ Top 15
# ------------------------------------------------------------
top15_importance = (
    permutation_importance_df
    .head(15)
    .sort_values(
        "Importance_Mean_WAPE_Increase",
        ascending=True
    )
)

plt.figure(figsize=(10, 7))

bars = plt.barh(
    top15_importance["Feature"],
    top15_importance[
        "Importance_Mean_WAPE_Increase"
    ]
)

plt.xlabel(
    "Increase in WAPE after permutation "
    "(percentage points)"
)

plt.ylabel("Feature")

plt.title(
    "Top 15 Features by Permutation Importance\n"
    "Final HGB Residual Rolling7 Pipeline"
)

# เส้นที่ค่า Importance = 0
plt.axvline(
    x=0,
    linewidth=1
)

# ใส่ตัวเลขปลายแท่ง
for bar in bars:

    value = bar.get_width()

    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:.3f}",
        va="center",
        fontsize=8
    )

plt.tight_layout()

importance_plot_path = os.path.join(
    OUTPUT_DIR,
    "final_pipeline_permutation_importance_top15.png"
)

plt.savefig(
    importance_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 7) แสดงผล
# ------------------------------------------------------------
print("\n" + "=" * 82)
print("Permutation Importance สำเร็จ")
print("=" * 82)

print("สูตร Pipeline ตรงกับ Final Prediction:", prediction_match)

print(
    "Base WAPE จาก Pipeline:",
    f"{base_test_wape:.2f}%"
)

print("\nTop 20 Features:")

display(
    permutation_importance_df
    .head(20)
    .style
    .format({
        "Importance_Mean_WAPE_Increase": "{:.4f}",
        "Importance_Std": "{:.4f}",
        "Min_WAPE_Increase": "{:.4f}",
        "Max_WAPE_Increase": "{:.4f}"
    })
)

print("\nบันทึกไฟล์แล้วที่:")

print(
    os.path.join(
        OUTPUT_DIR,
        "final_pipeline_permutation_importance.csv"
    )
)

print(importance_plot_path)

print(
    "\nหมายเหตุ: Importance แสดงความสำคัญต่อการพยากรณ์ "
    "ไม่ใช่ความสัมพันธ์เชิงเหตุและผล"
)

print(
    "ห้ามใช้ผล Feature Importance ย้อนกลับไปปรับ Final Model"
)

In [ ]:
# ============================================================
# STEP 17: Safety-buffer cash planning simulation
# ============================================================

# ------------------------------------------------------------
# 1) ระดับ Buffer ที่ต้องการทดลอง
# ------------------------------------------------------------
buffer_rates = [
    0.00,
    0.10,
    0.20,
    0.30
]

simulation_rows = []
simulation_detail_frames = []

actual_values = (
    final_pred_df["TARGET_T_PLUS_1"]
    .to_numpy(dtype=float)
)

base_prediction_values = (
    final_pred_df["PRED_FINAL"]
    .to_numpy(dtype=float)
)

total_records = len(final_pred_df)


# ------------------------------------------------------------
# 2) จำลองแต่ละระดับ Buffer
# ------------------------------------------------------------
for buffer_rate in buffer_rates:

    prepared_cash = (
        base_prediction_values
        * (1 + buffer_rate)
    )

    # เงินสดขาด: Actual มากกว่าเงินที่เตรียม
    shortage_amount = np.maximum(
        actual_values - prepared_cash,
        0
    )

    # เงินสดเกิน: เงินที่เตรียมมากกว่า Actual
    excess_amount = np.maximum(
        prepared_cash - actual_values,
        0
    )

    shortage_mask = shortage_amount > 0
    excess_mask = excess_amount > 0
    exact_mask = np.isclose(
        prepared_cash,
        actual_values,
        rtol=1e-10,
        atol=1e-6
    )

    shortage_records = shortage_mask.sum()
    excess_records = excess_mask.sum()
    exact_records = exact_mask.sum()

    total_shortage = shortage_amount.sum()
    total_excess = excess_amount.sum()

    # ค่าเฉลี่ยต่อ Branch-day ทั้งหมด
    avg_shortage_all_records = (
        total_shortage / total_records
    )

    avg_excess_all_records = (
        total_excess / total_records
    )

    # ค่าเฉลี่ยเฉพาะ Branch-day ที่เกิดเหตุการณ์
    avg_shortage_when_short = (
        shortage_amount[shortage_mask].mean()
        if shortage_records > 0
        else 0
    )

    avg_excess_when_excess = (
        excess_amount[excess_mask].mean()
        if excess_records > 0
        else 0
    )

    simulation_rows.append({
        "Buffer_Rate": buffer_rate,
        "Buffer_Percent": buffer_rate * 100,
        "Total_Branch_Day_Records": total_records,

        "Shortage_Records": shortage_records,
        "Shortage_Rate_Percent": (
            shortage_records / total_records * 100
        ),
        "Total_Shortage_Amount": total_shortage,
        "Average_Shortage_Per_All_Record": (
            avg_shortage_all_records
        ),
        "Average_Shortage_When_Occurred": (
            avg_shortage_when_short
        ),

        "Excess_Records": excess_records,
        "Excess_Rate_Percent": (
            excess_records / total_records * 100
        ),
        "Total_Excess_Amount": total_excess,
        "Average_Excess_Per_All_Record": (
            avg_excess_all_records
        ),
        "Average_Excess_When_Occurred": (
            avg_excess_when_excess
        ),

        "Exact_Records": exact_records,
        "Prepared_Cash_Total": prepared_cash.sum(),
        "Actual_Cash_Out_Total": actual_values.sum()
    })


    # เก็บรายละเอียดราย Branch-day
    detail = final_pred_df[
        [
            "EFFECTIVE_DATE",
            "FORECAST_DATE",
            "BRANCH_CODE",
            "TARGET_T_PLUS_1",
            "PRED_FINAL"
        ]
    ].copy()

    detail["BUFFER_RATE"] = buffer_rate
    detail["BUFFER_PERCENT"] = buffer_rate * 100
    detail["PREPARED_CASH"] = prepared_cash
    detail["SHORTAGE_AMOUNT"] = shortage_amount
    detail["EXCESS_AMOUNT"] = excess_amount
    detail["IS_SHORTAGE"] = shortage_mask.astype(int)
    detail["IS_EXCESS"] = excess_mask.astype(int)

    simulation_detail_frames.append(detail)


# ------------------------------------------------------------
# 3) รวมผล
# ------------------------------------------------------------
buffer_simulation_summary = pd.DataFrame(
    simulation_rows
)

buffer_simulation_detail = pd.concat(
    simulation_detail_frames,
    ignore_index=True
)


# ------------------------------------------------------------
# 4) ตรวจสอบสูตร
# ------------------------------------------------------------
if len(buffer_simulation_summary) != len(buffer_rates):
    raise ValueError(
        "จำนวนระดับ Buffer ในผล Simulation ไม่ตรง"
    )

if (
    buffer_simulation_summary[
        "Shortage_Records"
    ]
    + buffer_simulation_summary[
        "Excess_Records"
    ]
    + buffer_simulation_summary[
        "Exact_Records"
    ]
    != total_records
).any():
    raise ValueError(
        "จำนวน Shortage, Excess และ Exact รวมกันไม่ตรง"
    )

# เมื่อ Buffer เพิ่มขึ้น Shortage ควรไม่เพิ่ม
shortage_monotonic = (
    buffer_simulation_summary[
        "Total_Shortage_Amount"
    ]
    .diff()
    .dropna()
    .le(1e-8)
    .all()
)

# เมื่อ Buffer เพิ่มขึ้น Excess ควรไม่ลด
excess_monotonic = (
    buffer_simulation_summary[
        "Total_Excess_Amount"
    ]
    .diff()
    .dropna()
    .ge(-1e-8)
    .all()
)

if not shortage_monotonic:
    raise ValueError(
        "Buffer เพิ่มขึ้น แต่ยอด Shortage กลับเพิ่มขึ้น"
    )

if not excess_monotonic:
    raise ValueError(
        "Buffer เพิ่มขึ้น แต่ยอด Excess กลับลดลง"
    )


# ------------------------------------------------------------
# 5) บันทึกผล
# ------------------------------------------------------------
summary_path = os.path.join(
    OUTPUT_DIR,
    "safety_buffer_simulation_summary.csv"
)

detail_path = os.path.join(
    OUTPUT_DIR,
    "safety_buffer_simulation_detail.csv"
)

buffer_simulation_summary.to_csv(
    summary_path,
    index=False
)

buffer_simulation_detail.to_csv(
    detail_path,
    index=False
)


# ------------------------------------------------------------
# 6) กราฟ Simulated Under-coverage และ Over-coverage
# ------------------------------------------------------------

x_values = (
    buffer_simulation_summary[
        "Buffer_Percent"
    ].to_numpy()
)

under_coverage_rates = (
    buffer_simulation_summary[
        "Shortage_Rate_Percent"
    ].to_numpy()
)

over_coverage_rates = (
    buffer_simulation_summary[
        "Excess_Rate_Percent"
    ].to_numpy()
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    x_values,
    under_coverage_rates,
    marker="o",
    linewidth=2,
    label="Under-coverage rate"
)

ax.plot(
    x_values,
    over_coverage_rates,
    marker="o",
    linewidth=2,
    label="Over-coverage rate"
)

# แสดงเปอร์เซ็นต์ของ Under-coverage ใต้จุด
for x_value, y_value in zip(
    x_values,
    under_coverage_rates
):
    ax.annotate(
        f"{y_value:.2f}%",
        xy=(x_value, y_value),
        xytext=(0, -15),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=9
    )

# แสดงเปอร์เซ็นต์ของ Over-coverage เหนือจุด
for x_value, y_value in zip(
    x_values,
    over_coverage_rates
):
    ax.annotate(
        f"{y_value:.2f}%",
        xy=(x_value, y_value),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_xlabel(
    "Buffer Rate (%)"
)

ax.set_ylabel(
    "Branch-day Records (%)"
)

ax.set_title(
    "Simulated Under-coverage and Over-coverage by Buffer Rate"
)

ax.set_xticks(
    x_values
)

# เพิ่มพื้นที่ด้านบนเพื่อไม่ให้ตัวเลข 64.67% ถูกบัง
maximum_rate = max(
    under_coverage_rates.max(),
    over_coverage_rates.max()
)

ax.set_ylim(
    0,
    maximum_rate * 1.15
)

ax.legend(
    loc="best"
)

ax.grid(
    alpha=0.3
)

fig.tight_layout()

tradeoff_plot_path = os.path.join(
    OUTPUT_DIR,
    "safety_buffer_tradeoff.png"
)

fig.savefig(
    tradeoff_plot_path,
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)


# ------------------------------------------------------------
# 7) แสดงผล
# ------------------------------------------------------------
print("=" * 85)
print("จำลอง Safety Buffer สำเร็จ")
print("=" * 85)

print(
    "จำนวน Final Test branch-day records:",
    f"{total_records:,}"
)

print(
    "Shortage ลดลงต่อเนื่องเมื่อเพิ่ม Buffer:",
    shortage_monotonic
)

print(
    "Excess เพิ่มขึ้นต่อเนื่องเมื่อเพิ่ม Buffer:",
    excess_monotonic
)

print("\nผล Simulation:")

display(
    buffer_simulation_summary[
        [
            "Buffer_Percent",
            "Shortage_Records",
            "Shortage_Rate_Percent",
            "Total_Shortage_Amount",
            "Average_Shortage_Per_All_Record",
            "Average_Shortage_When_Occurred",
            "Excess_Records",
            "Excess_Rate_Percent",
            "Total_Excess_Amount",
            "Average_Excess_Per_All_Record",
            "Average_Excess_When_Occurred"
        ]
    ].style.format({
        "Buffer_Percent": "{:.0f}%",
        "Shortage_Rate_Percent": "{:.2f}%",
        "Total_Shortage_Amount": "{:,.2f}",
        "Average_Shortage_Per_All_Record": "{:,.2f}",
        "Average_Shortage_When_Occurred": "{:,.2f}",
        "Excess_Rate_Percent": "{:.2f}%",
        "Total_Excess_Amount": "{:,.2f}",
        "Average_Excess_Per_All_Record": "{:,.2f}",
        "Average_Excess_When_Occurred": "{:,.2f}"
    })
)

print("\nบันทึกไฟล์แล้วที่:")
print(summary_path)
print(detail_path)
print(tradeoff_plot_path)

print(
    "\nหมายเหตุ: ยังไม่สามารถระบุ Buffer ที่ดีที่สุดได้ "
    "จนกว่าจะทราบต้นทุนของเงินสดขาดและเงินสดส่วนเกิน"
)

In [ ]:
# ============================================================
# STEP 18: Final reporting, audit summary, and model export
# ============================================================

import json
import joblib


# ------------------------------------------------------------
# 1) สรุปผลการเลือกโมเดลบน Validation
# ------------------------------------------------------------
validation_baseline_report = (
    validation_baseline_results[
        ["Model", "MAE", "RMSE", "WAPE (%)"]
    ]
    .copy()
)

validation_baseline_report["Approach"] = "Baseline"


validation_direct_report = (
    validation_ml_results[
        ["Model", "MAE", "RMSE", "WAPE (%)"]
    ]
    .copy()
)

validation_direct_report["Approach"] = "Direct Machine Learning"


validation_residual_report = (
    residual_validation_results[
        ["Model", "MAE", "RMSE", "WAPE (%)"]
    ]
    .copy()
)

validation_residual_report["Approach"] = "Residual Machine Learning"


validation_model_selection = pd.concat(
    [
        validation_baseline_report,
        validation_direct_report,
        validation_residual_report
    ],
    ignore_index=True
)

validation_model_selection = (
    validation_model_selection
    .sort_values(
        ["WAPE (%)", "MAE"],
        ascending=True
    )
    .reset_index(drop=True)
)

validation_model_selection.insert(
    0,
    "Validation_Rank",
    np.arange(
        1,
        len(validation_model_selection) + 1
    )
)


# ------------------------------------------------------------
# 2) สรุปผล Final Test
# ------------------------------------------------------------
final_metrics_summary = pd.DataFrame([
    {
        "Dataset": "Validation",
        "Model": best_residual_candidate_id,
        "Records": len(y_validation),
        "Start_Date": (
            meta_validation["FORECAST_DATE"].min()
        ),
        "End_Date": (
            meta_validation["FORECAST_DATE"].max()
        ),
        "MAE": (
            residual_validation_results.iloc[0]["MAE"]
        ),
        "RMSE": (
            residual_validation_results.iloc[0]["RMSE"]
        ),
        "WAPE (%)": best_residual_validation_wape
    },
    {
        "Dataset": "Final Test",
        "Model": LOCKED_MODEL_NAME,
        "Records": len(y_test),
        "Start_Date": (
            meta_test["FORECAST_DATE"].min()
        ),
        "End_Date": (
            meta_test["FORECAST_DATE"].max()
        ),
        "MAE": final_test_result["MAE"],
        "RMSE": final_test_result["RMSE"],
        "WAPE (%)": final_test_result["WAPE (%)"]
    },
    {
        "Dataset": "Final Test",
        "Model": "Rolling Mean 7 Days",
        "Records": len(y_test),
        "Start_Date": (
            meta_test["FORECAST_DATE"].min()
        ),
        "End_Date": (
            meta_test["FORECAST_DATE"].max()
        ),
        "MAE": final_test_baseline_result["MAE"],
        "RMSE": final_test_baseline_result["RMSE"],
        "WAPE (%)": (
            final_test_baseline_result["WAPE (%)"]
        )
    }
])


# ------------------------------------------------------------
# 3) Target statistics ของข้อมูล Model-ready
# ------------------------------------------------------------
target_statistics = pd.DataFrame([
    {
        "Count": len(y),
        "Mean": y.mean(),
        "Standard_Deviation": y.std(),
        "Minimum": y.min(),
        "P25": y.quantile(0.25),
        "Median": y.quantile(0.50),
        "P75": y.quantile(0.75),
        "P95": y.quantile(0.95),
        "P99": y.quantile(0.99),
        "Maximum": y.max(),
        "Zero_Count": (y == 0).sum(),
        "Zero_Percent": (y == 0).mean() * 100
    }
])


# ------------------------------------------------------------
# 4) Actual vs Predicted รวมทุกสาขารายวัน
# ------------------------------------------------------------
daily_actual_predicted = (
    final_pred_df
    .groupby("FORECAST_DATE", as_index=False)
    .agg(
        ACTUAL_TOTAL=(
            "TARGET_T_PLUS_1",
            "sum"
        ),
        PREDICTED_TOTAL=(
            "PRED_FINAL",
            "sum"
        )
    )
)

daily_actual_predicted["ABSOLUTE_ERROR"] = np.abs(
    daily_actual_predicted["ACTUAL_TOTAL"]
    - daily_actual_predicted["PREDICTED_TOTAL"]
)


# ------------------------------------------------------------
# 5) กราฟ Validation Model Comparison
# ------------------------------------------------------------
top_validation_models = (
    validation_model_selection
    .head(12)
    .sort_values(
        "WAPE (%)",
        ascending=True
    )
)

plt.figure(figsize=(10, 7))

bars = plt.barh(
    top_validation_models["Model"],
    top_validation_models["WAPE (%)"]
)

plt.xlabel("Validation WAPE (%)")
plt.ylabel("Model")
plt.title(
    "Model Selection Based on Validation WAPE"
)

for bar in bars:
    value = bar.get_width()

    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:.2f}%",
        va="center",
        fontsize=8
    )

plt.tight_layout()

validation_plot_path = os.path.join(
    OUTPUT_DIR,
    "validation_model_comparison.png"
)

plt.savefig(
    validation_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 6) กราฟ Actual vs Predicted รายวัน
# ------------------------------------------------------------
plt.figure(figsize=(13, 6))

plt.plot(
    daily_actual_predicted["FORECAST_DATE"],
    daily_actual_predicted["ACTUAL_TOTAL"],
    label="Actual"
)

plt.plot(
    daily_actual_predicted["FORECAST_DATE"],
    daily_actual_predicted["PREDICTED_TOTAL"],
    label="Predicted"
)

plt.xlabel("Forecast Date")
plt.ylabel("Total Cash-out")
plt.title(
    "Daily Actual vs Predicted Cash-out\n"
    "Aggregated Across 10 Branches"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

actual_predicted_plot_path = os.path.join(
    OUTPUT_DIR,
    "final_test_actual_vs_predicted_daily.png"
)

plt.savefig(
    actual_predicted_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 7) กราฟ WAPE รายสาขา
# ------------------------------------------------------------
branch_plot_data = (
    branch_performance
    .sort_values(
        "WAPE (%)",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    branch_plot_data["BRANCH_CODE"],
    branch_plot_data["WAPE (%)"]
)

plt.xlabel("Final Test WAPE (%)")
plt.ylabel("Branch")
plt.title("Final Test WAPE by Branch")

for bar in bars:
    value = bar.get_width()

    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:.2f}%",
        va="center",
        fontsize=9
    )

plt.tight_layout()

branch_plot_path = os.path.join(
    OUTPUT_DIR,
    "final_test_wape_by_branch.png"
)

plt.savefig(
    branch_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 8) Final Methodology Audit
# ------------------------------------------------------------
methodology_audit = pd.DataFrame([
    {
        "Audit_Item": "Raw duplicate branch-date records",
        "Result": duplicate_count,
        "Pass": duplicate_count == 0
    },
    {
        "Audit_Item": "Target mismatch with direct date join",
        "Result": int(target_mismatch_count),
        "Pass": target_mismatch_count == 0
    },
    {
        "Audit_Item": "Rolling30 calculation mismatch",
        "Result": int(rolling30_mismatch),
        "Pass": rolling30_mismatch == 0
    },
    {
        "Audit_Item": "Lag1 calculation mismatch",
        "Result": int(lag1_mismatch),
        "Pass": lag1_mismatch == 0
    },
    {
        "Audit_Item": "Future-date feature violation",
        "Result": int(future_data_violation),
        "Pass": future_data_violation == 0
    },
    {
        "Audit_Item": "Subtrain-validation date overlap",
        "Result": len(overlap_subtrain_validation),
        "Pass": len(overlap_subtrain_validation) == 0
    },
    {
        "Audit_Item": "Subtrain-test date overlap",
        "Result": len(overlap_subtrain_test),
        "Pass": len(overlap_subtrain_test) == 0
    },
    {
        "Audit_Item": "Validation-test date overlap",
        "Result": len(overlap_validation_test),
        "Pass": len(overlap_validation_test) == 0
    },
    {
        "Audit_Item": "Final pipeline prediction match",
        "Result": bool(prediction_match),
        "Pass": bool(prediction_match)
    },
    {
        "Audit_Item": "Branch WAPE reconstructs overall WAPE",
        "Result": float(overall_wape_difference),
        "Pass": overall_wape_difference < 1e-8
    },
    {
        "Audit_Item": "Shortage decreases with buffer",
        "Result": bool(shortage_monotonic),
        "Pass": bool(shortage_monotonic)
    },
    {
        "Audit_Item": "Excess increases with buffer",
        "Result": bool(excess_monotonic),
        "Pass": bool(excess_monotonic)
    }
])

all_audits_passed = (
    methodology_audit["Pass"].all()
)


# ------------------------------------------------------------
# 9) บันทึก Final Model และ Metadata
# ------------------------------------------------------------
final_model_path = os.path.join(
    OUTPUT_DIR,
    "final_hgb_residual_model.joblib"
)

joblib.dump(
    final_model,
    final_model_path
)


model_metadata = {
    "model_name": LOCKED_MODEL_NAME,
    "model_class": (
        final_model.__class__.__name__
    ),
    "target": "TARGET_T_PLUS_1",
    "forecast_horizon": "Next calendar day",
    "baseline_feature": final_baseline_col,
    "clip_quantile": float(
        LOCKED_CLIP_QUANTILE
    ),
    "residual_lower_bound": float(
        final_residual_lower_bound
    ),
    "residual_upper_bound": float(
        final_residual_upper_bound
    ),
    "random_state": int(RANDOM_STATE),
    "number_of_features": int(X.shape[1]),
    "feature_columns": X.columns.tolist(),
    "development_records": int(
        len(X_development)
    ),
    "final_test_records": int(
        len(X_test)
    ),
    "final_test_start": str(
        meta_test["FORECAST_DATE"].min().date()
    ),
    "final_test_end": str(
        meta_test["FORECAST_DATE"].max().date()
    ),
    "validation_wape_percent": float(
        best_residual_validation_wape
    ),
    "final_test_mae": float(
        final_test_result["MAE"]
    ),
    "final_test_rmse": float(
        final_test_result["RMSE"]
    ),
    "final_test_wape_percent": float(
        final_test_result["WAPE (%)"]
    ),
    "baseline_test_wape_percent": float(
        final_test_baseline_result["WAPE (%)"]
    )
}

metadata_path = os.path.join(
    OUTPUT_DIR,
    "final_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_metadata,
        file,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 10) บันทึกตารางทั้งหมด
# ------------------------------------------------------------
validation_model_selection.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_model_selection.csv"
    ),
    index=False
)

final_metrics_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_metrics_summary.csv"
    ),
    index=False
)

target_statistics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "model_ready_target_statistics.csv"
    ),
    index=False
)

daily_actual_predicted.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_test_daily_actual_predicted.csv"
    ),
    index=False
)

methodology_audit.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "methodology_audit.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 11) แสดงผลสรุป
# ------------------------------------------------------------
print("=" * 86)
print("จัดทำ Final Report และบันทึกโมเดลสำเร็จ")
print("=" * 86)

print("\nFinal Model:")
print(LOCKED_MODEL_NAME)

print(
    "Validation WAPE:",
    f"{best_residual_validation_wape:.2f}%"
)

print(
    "Final Test WAPE:",
    f"{final_model_wape:.2f}%"
)

print(
    "Final Test MAE:",
    f"{final_test_result['MAE']:,.2f}"
)

print(
    "Final Test RMSE:",
    f"{final_test_result['RMSE']:,.2f}"
)

print(
    "Baseline Final Test WAPE:",
    f"{final_baseline_wape:.2f}%"
)

print(
    "WAPE Improvement:",
    f"{final_wape_improvement:.2f}",
    "percentage points"
)

print("\nTarget Statistics:")
display(
    target_statistics.style.format({
        "Mean": "{:,.2f}",
        "Standard_Deviation": "{:,.2f}",
        "Minimum": "{:,.2f}",
        "P25": "{:,.2f}",
        "Median": "{:,.2f}",
        "P75": "{:,.2f}",
        "P95": "{:,.2f}",
        "P99": "{:,.2f}",
        "Maximum": "{:,.2f}",
        "Zero_Percent": "{:.2f}%"
    })
)

print("\nMethodology Audit:")
display(methodology_audit)

print(
    "\nทุก Audit ผ่าน:",
    all_audits_passed
)

if not all_audits_passed:
    raise ValueError(
        "ยังมี Methodology Audit บางรายการไม่ผ่าน"
    )

print("\nไฟล์สำคัญถูกบันทึกใน:", OUTPUT_DIR)
print(final_model_path)
print(metadata_path)
print(validation_plot_path)
print(actual_predicted_plot_path)
print(branch_plot_path)

print(
    "\nสถานะ: โมเดลและผลลัพธ์ถูกล็อกเรียบร้อย "
    "ไม่ควรปรับโมเดลเพิ่มเติมจาก Final Test"
)

In [ ]:
# ============================================================
# STEP 19: Package all final outputs for download
# ============================================================

import os
import shutil
from google.colab import files

# ------------------------------------------------------------
# 1) ตรวจว่าโฟลเดอร์ผลลัพธ์มีอยู่จริง
# ------------------------------------------------------------
if not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(
        f"ไม่พบโฟลเดอร์ {OUTPUT_DIR}"
    )

output_files = sorted(
    os.listdir(OUTPUT_DIR)
)

if len(output_files) == 0:
    raise ValueError(
        f"โฟลเดอร์ {OUTPUT_DIR} ไม่มีไฟล์ผลลัพธ์"
    )


# ------------------------------------------------------------
# 2) แสดงรายชื่อไฟล์ทั้งหมด
# ------------------------------------------------------------
print("=" * 80)
print("รายการไฟล์ Final Output")
print("=" * 80)

for number, file_name in enumerate(
    output_files,
    start=1
):
    file_path = os.path.join(
        OUTPUT_DIR,
        file_name
    )

    file_size_mb = (
        os.path.getsize(file_path)
        / (1024 ** 2)
    )

    print(
        f"{number:>2}. "
        f"{file_name:<55} "
        f"{file_size_mb:>8.3f} MB"
    )


# ------------------------------------------------------------
# 3) สร้างไฟล์ README สรุปผล
# ------------------------------------------------------------
readme_path = os.path.join(
    OUTPUT_DIR,
    "README_FINAL_RESULTS.txt"
)

readme_content = f"""
FINAL IS MODEL RESULTS
======================

Final Model:
{LOCKED_MODEL_NAME}

Forecast Horizon:
Next calendar day (T+1)

Model Selection:
Selected using chronological Validation set only.

Validation WAPE:
{best_residual_validation_wape:.2f}%

Final Test Period:
{meta_test["FORECAST_DATE"].min().date()} to
{meta_test["FORECAST_DATE"].max().date()}

Final Test Records:
{len(y_test):,} branch-day records

Final Test MAE:
{final_test_result["MAE"]:,.2f} THB

Final Test RMSE:
{final_test_result["RMSE"]:,.2f} THB

Final Test WAPE:
{final_test_result["WAPE (%)"]:.2f}%

Rolling7 Baseline WAPE:
{final_test_baseline_result["WAPE (%)"]:.2f}%

WAPE Improvement:
{final_wape_improvement:.2f} percentage points

Best Branch:
{best_branch["BRANCH_CODE"]}
WAPE = {best_branch["WAPE (%)"]:.2f}%

Highest-WAPE Branch:
{worst_branch["BRANCH_CODE"]}
WAPE = {worst_branch["WAPE (%)"]:.2f}%

Important Predictive Features:
1. TOTAL_CASH_OUT_ROLLING7
2. FORECAST_IS_WEEKEND
3. FORECAST_IS_HOLIDAY
4. Branch indicators
5. Historical cash-out variability and cash stock

Important Methodology Notes:
- Chronological split
- Model selected on Validation
- Final Test evaluated once
- No future-date leakage detected
- Missing branch-date records were not replaced with zero
- Feature importance is predictive, not causal
- Safety-buffer simulation uses branch-day records
- No single optimal buffer can be selected without business cost weights

All methodology audits passed:
{all_audits_passed}
"""

with open(
    readme_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(readme_content.strip())


# ------------------------------------------------------------
# 4) สร้าง ZIP
# ------------------------------------------------------------
zip_base_name = "IS_FINAL_STRICT_RESULTS"

zip_path = shutil.make_archive(
    base_name=zip_base_name,
    format="zip",
    root_dir=OUTPUT_DIR
)


# ------------------------------------------------------------
# 5) ตรวจสอบ ZIP
# ------------------------------------------------------------
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        "สร้างไฟล์ ZIP ไม่สำเร็จ"
    )

zip_size_mb = (
    os.path.getsize(zip_path)
    / (1024 ** 2)
)


print("\n" + "=" * 80)
print("สร้างไฟล์ ZIP สำเร็จ")
print("=" * 80)

print("ชื่อไฟล์:", zip_path)
print("ขนาด:", f"{zip_size_mb:.2f} MB")
print(
    "จำนวนไฟล์ในโฟลเดอร์:",
    len(os.listdir(OUTPUT_DIR))
)

print("\nกำลังดาวน์โหลดไฟล์ ZIP...")

files.download(zip_path)

In [ ]:
# เลือกเฉพาะโมเดลตัวแทนที่ควรแสดงในสไลด์
slide_model_order = [
    "Ridge Regression",
    "Naive: Current Day Cash-out",
    "Rolling Mean 7 Days",
    "Random Forest",
    "XGBoost",
    "Extra Trees",
    "HistGradientBoosting",
    "HGB_Residual_Rolling7_NoClip",
    "HGB_Residual_Rolling7_Clip_1.0pct"
]

slide_model_results = (
    validation_model_selection[
        validation_model_selection["Model"].isin(slide_model_order)
    ]
    .copy()
)

# เปลี่ยนชื่อให้สั้นและอ่านง่าย
name_map = {
    "Naive: Current Day Cash-out": "Naive: Current Day",
    "Rolling Mean 7 Days": "Rolling Mean 7 Days",
    "HistGradientBoosting": "HGB Direct",
    "HGB_Residual_Rolling7_NoClip": "HGB Residual Rolling7",
    "HGB_Residual_Rolling7_Clip_1.0pct":
        "HGB Residual Rolling7 + Clip 1%"
}

slide_model_results["Display_Name"] = (
    slide_model_results["Model"]
    .replace(name_map)
)

slide_model_results = slide_model_results.sort_values(
    "WAPE (%)",
    ascending=False
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    slide_model_results["Display_Name"],
    slide_model_results["WAPE (%)"]
)

plt.xlabel("Validation WAPE (%)")
plt.ylabel("Model")
plt.title("Model Comparison and Improvement on Validation Set")

for bar in bars:
    value = bar.get_width()
    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:.2f}%",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(
    "strict_outputs/validation_model_comparison_for_slide.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# อ่านผล Final Test ที่บันทึกไว้
example_df = pd.read_csv(
    "strict_outputs/final_test_predictions.csv"
)

# ตรวจชื่อคอลัมน์
print(example_df.columns.tolist())

# แปลงวันที่
for col in ["EFFECTIVE_DATE", "FORECAST_DATE"]:
    example_df[col] = pd.to_datetime(example_df[col])

# เลือกเฉพาะสาขา C1 และวันที่ค่าจริงมากกว่า 0
c1 = example_df[
    (example_df["BRANCH_CODE"] == "C1") &
    (example_df["TARGET_T_PLUS_1"] > 0)
].copy()

# ถ้ายังไม่มี ABSOLUTE_ERROR ให้สร้าง
if "ABSOLUTE_ERROR" not in c1.columns:
    c1["ABSOLUTE_ERROR"] = np.abs(
        c1["TARGET_T_PLUS_1"] - c1["PRED_FINAL"]
    )

# เลือกแถวที่ Absolute Error ใกล้ค่ามัธยฐานที่สุด
median_error = c1["ABSOLUTE_ERROR"].median()

example_row = c1.loc[
    (c1["ABSOLUTE_ERROR"] - median_error).abs().idxmin()
].copy()

# คำนวณเงินที่ควรเตรียมตาม Buffer
example_row["BUFFER_10"] = example_row["PRED_FINAL"] * 1.10
example_row["BUFFER_20"] = example_row["PRED_FINAL"] * 1.20
example_row["BUFFER_30"] = example_row["PRED_FINAL"] * 1.30

# แสดงผลเป็นล้านบาท
summary = pd.DataFrame({
    "รายการ": [
        "วันที่ใช้ข้อมูล (t)",
        "วันที่พยากรณ์ (t+1)",
        "Rolling7 Baseline",
        "Predicted Residual",
        "Final Prediction",
        "Actual Cash-out",
        "Absolute Error",
        "Prepared Cash + Buffer 10%",
        "Prepared Cash + Buffer 20%",
        "Prepared Cash + Buffer 30%"
    ],
    "ค่า": [
        example_row["EFFECTIVE_DATE"].date(),
        example_row["FORECAST_DATE"].date(),
        f'{example_row["ROLLING7_BASELINE"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["PREDICTED_RESIDUAL_CLIPPED"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["PRED_FINAL"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["TARGET_T_PLUS_1"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["ABSOLUTE_ERROR"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["BUFFER_10"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["BUFFER_20"]/1_000_000:.2f} ล้านบาท',
        f'{example_row["BUFFER_30"]/1_000_000:.2f} ล้านบาท'
    ]
})

display(summary)

In [ ]:
import sys
import os
import platform
import psutil
import numpy
import pandas
import sklearn
import xgboost
import matplotlib
import holidays

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Matplotlib:", matplotlib.__version__)
print("holidays:", holidays.__version__)
print("Processor:", platform.processor())
print("CPU cores:", os.cpu_count())
print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))

#รูปใส่ word

In [ ]:
# ============================================================
# REPORT FIGURES ONLY
# Run this cell after completing the original notebook
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1) ตรวจสอบตัวแปรจากโค้ดเดิม
# ------------------------------------------------------------
required_variables = [
    "final_pred_df",
    "branch_performance",
    "permutation_importance_df",
    "buffer_simulation_summary"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "กรุณารันโค้ดเดิมให้จบก่อน ตัวแปรที่ยังไม่มี: "
        + ", ".join(missing_variables)
    )


# ------------------------------------------------------------
# 2) สร้างโฟลเดอร์สำหรับรูปที่ใช้ในรายงาน
# ------------------------------------------------------------
REPORT_FIGURE_DIR = os.path.join(
    OUTPUT_DIR,
    "report_figures"
)

os.makedirs(
    REPORT_FIGURE_DIR,
    exist_ok=True
)


def save_report_figure(fig, file_name):
    """
    บันทึกทั้ง PNG สำหรับใส่ Word
    และ SVG สำหรับไฟล์แบบ Vector
    """

    png_path = os.path.join(
        REPORT_FIGURE_DIR,
        file_name + ".png"
    )

    svg_path = os.path.join(
        REPORT_FIGURE_DIR,
        file_name + ".svg"
    )

    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.20
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight",
        pad_inches=0.20
    )

    return png_path, svg_path


saved_report_figures = []


# ============================================================
# FIGURE 1: Daily Actual vs Predicted Cash-out
# ============================================================

report_daily = (
    final_pred_df
    .groupby(
        "FORECAST_DATE",
        as_index=False
    )
    .agg(
        ACTUAL_TOTAL=(
            "TARGET_T_PLUS_1",
            "sum"
        ),
        PREDICTED_TOTAL=(
            "PRED_FINAL",
            "sum"
        )
    )
)

fig, ax = plt.subplots(
    figsize=(13, 6)
)

ax.plot(
    report_daily["FORECAST_DATE"],
    report_daily["ACTUAL_TOTAL"],
    label="Actual",
    linewidth=1.3
)

ax.plot(
    report_daily["FORECAST_DATE"],
    report_daily["PREDICTED_TOTAL"],
    label="Predicted",
    linewidth=1.3
)

ax.set_xlabel("Forecast Date")
ax.set_ylabel("Total Cash-out (THB)")

ax.set_title(
    "Daily Actual vs Predicted Cash-out\n"
    "Aggregated Across 10 Branches"
)

ax.legend()
ax.grid(alpha=0.30)

fig.autofmt_xdate()
fig.tight_layout()

saved_report_figures.append(
    save_report_figure(
        fig,
        "fig_actual_vs_predicted_final_test"
    )
)

plt.show()
plt.close(fig)


# ============================================================
# FIGURE 2: Final Test WAPE by Branch
# แก้ตัวเลข 144.16% ไม่ให้ถูกบัง
# ============================================================

report_branch = (
    branch_performance
    .sort_values(
        "WAPE (%)",
        ascending=True
    )
    .copy()
)

fig, ax = plt.subplots(
    figsize=(10.5, 6.5)
)

bars = ax.barh(
    report_branch["BRANCH_CODE"],
    report_branch["WAPE (%)"]
)

ax.set_xlabel("Final Test WAPE (%)")
ax.set_ylabel("Branch")
ax.set_title("Final Test WAPE by Branch")

branch_max = report_branch["WAPE (%)"].max()
branch_padding = max(10, branch_max * 0.13)

ax.set_xlim(
    0,
    branch_max + branch_padding
)

for bar in bars:

    value = bar.get_width()

    ax.annotate(
        f"{value:.2f}%",
        xy=(
            value,
            bar.get_y() + bar.get_height() / 2
        ),
        xytext=(5, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9,
        clip_on=False
    )

ax.grid(
    axis="x",
    alpha=0.20
)

fig.subplots_adjust(
    left=0.13,
    right=0.96,
    top=0.90,
    bottom=0.13
)

saved_report_figures.append(
    save_report_figure(
        fig,
        "fig_final_test_wape_by_branch"
    )
)

plt.show()
plt.close(fig)


# ============================================================
# FIGURE 3: Top 15 Permutation Importance
# แก้ตัวเลข 36.535 ไม่ให้ถูกบัง
# ============================================================

report_importance = (
    permutation_importance_df
    .head(15)
    .sort_values(
        "Importance_Mean_WAPE_Increase",
        ascending=True
    )
    .copy()
)

fig, ax = plt.subplots(
    figsize=(11.5, 7.2)
)

bars = ax.barh(
    report_importance["Feature"],
    report_importance[
        "Importance_Mean_WAPE_Increase"
    ]
)

ax.set_xlabel(
    "Increase in WAPE after Permutation "
    "(Percentage Points)"
)

ax.set_ylabel("Feature")

ax.set_title(
    "Top 15 Features by Permutation Importance\n"
    "Final HGB Residual Rolling7 Pipeline"
)

ax.axvline(
    x=0,
    color="black",
    linewidth=0.8
)

importance_values = report_importance[
    "Importance_Mean_WAPE_Increase"
]

importance_min = min(
    0,
    importance_values.min()
)

importance_max = max(
    0,
    importance_values.max()
)

importance_range = max(
    importance_max - importance_min,
    1
)

ax.set_xlim(
    importance_min - importance_range * 0.02,
    importance_max + importance_range * 0.15
)

for bar in bars:

    value = bar.get_width()

    if value >= 0:
        text_offset = 5
        text_alignment = "left"
    else:
        text_offset = -5
        text_alignment = "right"

    ax.annotate(
        f"{value:.3f}",
        xy=(
            value,
            bar.get_y() + bar.get_height() / 2
        ),
        xytext=(text_offset, 0),
        textcoords="offset points",
        ha=text_alignment,
        va="center",
        fontsize=8,
        clip_on=False
    )

ax.grid(
    axis="x",
    alpha=0.20
)

fig.subplots_adjust(
    left=0.34,
    right=0.96,
    top=0.88,
    bottom=0.13
)

saved_report_figures.append(
    save_report_figure(
        fig,
        "fig_permutation_importance_top15"
    )
)

plt.show()
plt.close(fig)


# ============================================================
# FIGURE 4: Safety-buffer Trade-off
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    buffer_simulation_summary["Buffer_Percent"],
    buffer_simulation_summary[
        "Shortage_Rate_Percent"
    ],
    marker="o",
    linewidth=1.8,
    label="Shortage rate"
)

ax.plot(
    buffer_simulation_summary["Buffer_Percent"],
    buffer_simulation_summary[
        "Excess_Rate_Percent"
    ],
    marker="o",
    linewidth=1.8,
    label="Excess rate"
)

# แสดงตัวเลขแต่ละจุด
for _, row in buffer_simulation_summary.iterrows():

    ax.annotate(
        f'{row["Shortage_Rate_Percent"]:.2f}%',
        xy=(
            row["Buffer_Percent"],
            row["Shortage_Rate_Percent"]
        ),
        xytext=(0, -15),
        textcoords="offset points",
        ha="center",
        fontsize=8
    )

    ax.annotate(
        f'{row["Excess_Rate_Percent"]:.2f}%',
        xy=(
            row["Buffer_Percent"],
            row["Excess_Rate_Percent"]
        ),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=8
    )

ax.set_xlabel("Safety Buffer (%)")
ax.set_ylabel("Branch-day Record Rate (%)")

ax.set_title(
    "Trade-off between Cash Shortage and Excess"
)

ax.set_xticks(
    buffer_simulation_summary["Buffer_Percent"]
)

maximum_rate = max(
    buffer_simulation_summary[
        "Shortage_Rate_Percent"
    ].max(),
    buffer_simulation_summary[
        "Excess_Rate_Percent"
    ].max()
)

ax.set_ylim(
    0,
    maximum_rate * 1.15
)

ax.legend()
ax.grid(alpha=0.30)

fig.tight_layout()

saved_report_figures.append(
    save_report_figure(
        fig,
        "fig_safety_buffer_tradeoff"
    )
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# แสดงตำแหน่งไฟล์
# ------------------------------------------------------------

print("=" * 70)
print("สร้างเฉพาะรูปสำหรับรายงานเรียบร้อยแล้ว")
print("=" * 70)

for png_path, svg_path in saved_report_figures:
    print("PNG:", png_path)
    print("SVG:", svg_path)
    print()

In [ ]:
# ============================================================
# Revised Figure 5: Under-coverage and Over-coverage
# ============================================================

import os
import matplotlib.pyplot as plt

x = buffer_simulation_summary["Buffer_Percent"]
under_rate = buffer_simulation_summary["Shortage_Rate_Percent"]
over_rate = buffer_simulation_summary["Excess_Rate_Percent"]

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    x,
    under_rate,
    marker="o",
    linewidth=2,
    label="Under-coverage rate"
)

ax.plot(
    x,
    over_rate,
    marker="o",
    linewidth=2,
    label="Over-coverage rate"
)

# แสดงตัวเลขบนกราฟ
for buffer, value in zip(x, under_rate):
    ax.annotate(
        f"{value:.2f}%",
        (buffer, value),
        xytext=(0, -14),
        textcoords="offset points",
        ha="center"
    )

for buffer, value in zip(x, over_rate):
    ax.annotate(
        f"{value:.2f}%",
        (buffer, value),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center"
    )

ax.set_xlabel("Safety Buffer (%)")
ax.set_ylabel("Branch-day Record Rate (%)")
ax.set_title("Trade-off between Under-coverage and Over-coverage")
ax.set_xticks(x)
ax.set_ylim(0, max(over_rate) + 10)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

revised_plot_path = os.path.join(
    OUTPUT_DIR,
    "safety_buffer_tradeoff_revised.png"
)

plt.savefig(
    revised_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("บันทึกรูปใหม่ที่:", revised_plot_path)